# Data Pipeline For all 14 Datasets

In [69]:

import pandas as pd
import re
import os
import copy

# STEP 0: LOAD ALL 14 DATASETS
PARQUET = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/raw_data"

hpa_rna         = pd.read_parquet(f"{PARQUET}/1_4_hpa_rna_celline.parquet")
depmap_expr     = pd.read_parquet(f"{PARQUET}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.parquet")
geo_expr        = pd.read_parquet(f"{PARQUET}/3_GEOexpression.parquet")
proteomics      = pd.read_parquet(f"{PARQUET}/4_Harmonized_MS_CCLE_Gygi_subsetted.parquet")
fusions         = pd.read_parquet(f"{PARQUET}/5_OmicsFusionFilteredSupplementary.parquet")
mutations       = pd.read_parquet(f"{PARQUET}/6_OmicsSomaticMutationsProfile.parquet")
cellosaurus     = pd.read_parquet(f"{PARQUET}/7_cellosaurus.parquet")
depmap_profiles = pd.read_parquet(f"{PARQUET}/8_DepMap_OmicsProfiles.parquet")
sample_info     = pd.read_parquet(f"{PARQUET}/9_DepMap_sample_info.parquet")
geo_info        = pd.read_parquet(f"{PARQUET}/10_GEOInfo.parquet")
hpa_desc        = pd.read_parquet(f"{PARQUET}/11_hpa_rna_celline_description.parquet")
metabolomics    = pd.read_parquet(f"{PARQUET}/12_CCLE_metabolomics_20190502.parquet")
mirna           = pd.read_parquet(f"{PARQUET}/13_CCLE_miRNA_20181103.parquet")
signatures      = pd.read_parquet(f"{PARQUET}/14_OmicsGlobalSignatures.parquet")

tables = {
    "hpa_rna": hpa_rna,
    "depmap_expr": depmap_expr,
    "geo_expr": geo_expr,
    "proteomics": proteomics,
    "fusions": fusions,
    "mutations": mutations,
    "cellosaurus": cellosaurus,
    "depmap_profiles": depmap_profiles,
    "sample_info": sample_info,
    "geo_info": geo_info,
    "hpa_desc": hpa_desc,
    "metabolomics": metabolomics,
    "mirna": mirna,
    "signatures": signatures,
}

# Keep an untouched copy for before/after comparisons
tables_raw = copy.deepcopy(tables)

## 1. HPA_RNA

In [70]:
def clean_hpa_rna(df):
    """
    Clean the HPA RNA cell line table (hpa_rna).

    Steps:
        1. Lowercase all column names
        2. Apply explicit old->new column name mapping
        3. Lowercase all string values
        4. Strip + collapse extra whitespace in string values
        5. In 'gene' column, strip Ensembl version suffix
           (e.g. 'ensg00000000003.15' -> 'ensg00000000003')
        6. Normalise 'cell line' name in place -> remove separators
           (e.g. '537-mel' -> '537mel')

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).lower() for c in df.columns]

    # --- Step 2: explicit column rename mapping ---
    rename_map = {
        "gene": "gene",
        "gene name": "gene name",
        "cell line": "cell line",
        "tpm": "tpm",
        "ptpm": "ptpm",
        "ntpm": "ntpm",
    }
    df = df.rename(columns=rename_map)

    # --- Step 3 & 4: lowercase values + strip/collapse whitespace ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 5: strip Ensembl version suffix in 'gene' column ---
    if "gene" in df.columns:
        ens_pattern = r"(ens[gtp]\d+)\.\d+"
        df["gene"] = df["gene"].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df["gene"] = df["gene"].replace("nan", pd.NA)

    # --- Step 6: normalise 'cell line' name IN PLACE ---
    # remove separators (spaces, hyphens, dots, underscores): '537-mel' -> '537mel'
    if "cell line" in df.columns:
        df["cell line"] = (
            df["cell line"].astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"[\s\-\.\_]", "", regex=True)
        )
        df["cell line"] = df["cell line"].replace({"nan": pd.NA, "": pd.NA})

    return df

In [71]:
hpa_rna_clean = clean_hpa_rna(hpa_rna)
hpa_rna_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_rna_clean.parquet", index=False)

In [72]:
print(hpa_rna_clean)

                     gene gene name cell line   tpm  ptpm  ntpm
0         ensg00000000003    tspan6      143b  22.0  27.6  25.9
1         ensg00000000003    tspan6     22rv1   2.8   3.6   2.7
2         ensg00000000003    tspan6  23132/87   6.2   7.5   7.5
3         ensg00000000003    tspan6      253j  14.2  18.7  25.4
4         ensg00000000003    tspan6    253jbv  13.0  17.1  18.5
...                   ...       ...       ...   ...   ...   ...
24315367  ensg00000291317   tmem276      yh13  16.8  21.8  15.6
24315368  ensg00000291317   tmem276      ykg1  25.1  32.2  24.0
24315369  ensg00000291317   tmem276      ymb1  42.4  55.9  47.8
24315370  ensg00000291317   tmem276     zr751  30.0  40.9  30.8
24315371  ensg00000291317   tmem276    zr7530  43.1  53.8  65.7

[24315372 rows x 6 columns]


## 2. Depmap_Expr

In [73]:
def clean_transpose_depmap_expr(df):
    """
    Clean + transpose the DepMap expression table (depmap_expr).

    Input structure:  rows = PR- profile IDs (index),
                      columns = gene headers like 'TSPAN6 (ENSG00000000003)'.

    Output structure: rows = genes,
                      columns = ['gene', 'ensg_id', <PR- sample columns...>]

    Steps:
        1. Lowercase + clean the PR- index (sample IDs)
        2. Transpose so genes become rows, PR- IDs become columns
        3. Split each gene header into:
             - gene    (symbol, e.g. 'tspan6'; NA if header was bare ensg)
             - ensg_id (bare ensembl id, version-stripped)
        4. Strip Ensembl version suffix (.15 etc.)

    Returns:
        pd.DataFrame
    """
    df = df.copy()

    ens_core    = re.compile(r"ens[gtp]\d+", re.IGNORECASE)
    ens_version = re.compile(r"(ens[gtp]\d+)\.\d+", re.IGNORECASE)

    # --- Step 1: clean the PR- index (sample IDs) ---
    df.index = (
        df.index.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )
    df.index.name = "sample_id"

    # --- Step 2: transpose -> genes become rows, PR- IDs become columns ---
    dft = df.T
    dft.index.name = "gene_header"        # original header, e.g. 'tspan6 (ensg00000000003)'
    dft = dft.reset_index()

    # --- Step 3 & 4: split header into gene symbol + ensg_id ---
    def split_header(h):
        h = str(h).strip().lower()
        h = re.sub(r"\s+", " ", h)

        m = ens_core.search(h)
        ensg = ens_version.sub(r"\1", m.group(0)) if m else pd.NA   # version-stripped

        # gene symbol = text before the '(' (if present), else NA when header is bare ensg
        if "(" in h:
            sym = h.split("(")[0].strip()
            sym = sym if sym else pd.NA
        else:
            # header is just the ensg id (no symbol) -> no symbol available
            sym = pd.NA if (m and h == m.group(0)) else h

        return pd.Series({"gene": sym, "ensg_id": ensg})

    split_cols = dft["gene_header"].apply(split_header)

    # --- assemble: gene | ensg_id | <PR- sample columns...> ---
    sample_cols = [c for c in dft.columns if c != "gene_header"]
    out = pd.concat([split_cols, dft[sample_cols]], axis=1)

    return out

In [74]:
depmap_expr_clean = clean_transpose_depmap_expr(depmap_expr)
depmap_expr_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_expr_clean.parquet", index=False)

In [75]:
depmap_expr_clean

,gene,ensg_id,pr-adbjpg,pr-i2azwg,pr-5ekaac,pr-i21681,pr-i9drp1,pr-llpkng,pr-fesgd6,pr-z36vet,...,pr-mecfqo,pr-gf1lzy,pr-2lppyq,pr-iueft6,pr-heyoh9,pr-acnzor,pr-ez3iv8,pr-rwnv81,pr-ivfp8s,pr-asipq0
0,tspan6,ensg00000000003,4.331992,4.567424,3.150560,5.085340,6.729417,4.272770,3.337711,0.056584,...,6.345361,4.018812,4.328406,5.995032,3.533563,0.056584,3.111031,4.390943,5.057450,4.249445
1,tnmd,ensg00000000005,0.000000,0.584963,0.000000,0.000000,0.000000,0.189034,0.000000,0.000000,...,3.401903,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,dpm1,ensg00000000419,7.364660,7.106641,7.379118,7.154211,6.537917,7.023255,5.927659,6.094236,...,7.211694,6.700856,7.059182,6.238978,6.488483,6.604368,7.031329,7.013239,7.815191,6.175724
3,scyl3,ensg00000000457,2.792855,2.543496,2.333424,2.545968,2.456806,2.555816,1.944858,3.971773,...,2.533563,2.137504,1.891419,2.304511,1.823749,3.266037,1.541019,1.887525,2.538538,2.319040
4,c1orf112,ensg00000000460,4.471187,3.504620,4.228049,3.084064,3.867896,3.841973,2.678072,3.731183,...,4.374344,2.531069,3.529821,4.000000,3.308885,4.973152,3.664483,3.252476,3.893362,3.825786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53956,NaN,ensg00000288721,0.992768,0.432959,0.367371,0.411426,0.678072,0.189034,0.422233,1.304511,...,1.526069,0.411426,0.505891,0.925999,0.879706,1.244887,0.454176,0.695994,0.604071,0.985500
53957,NaN,ensg00000288722,2.797013,2.972693,1.695994,3.921246,4.418190,3.054848,0.201634,5.596637,...,3.307429,4.056584,3.821710,2.060047,4.978196,4.553975,5.377818,4.456806,4.196135,4.076388
53958,NaN,ensg00000288723,0.000000,0.056584,0.084064,0.028569,0.000000,0.000000,0.000000,0.124328,...,0.000000,0.070389,0.000000,0.000000,0.028569,0.056584,0.310340,0.367371,0.084064,0.000000
53959,NaN,ensg00000288724,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 3. Geo_Expr

In [76]:
def clean_geo_expr(df):
    """
    Clean the GEO expression table (geo_expr).

    Structure: rows = genes, columns = ['gene', 'GSM101610', 'GSM101615', ...]

    Steps:
        1. Lowercase column names + all string values
        2. Strip/collapse extra whitespace
        3. In 'gene' column, strip Ensembl version suffix
           ('ensg00000000003.15' -> 'ensg00000000003',
            'ensg00000000005.0'  -> 'ensg00000000005')

    Parameters:
        df (pd.DataFrame): raw geo_expr table

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 1 & 2: lowercase values + strip/collapse whitespace (string cols only) ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: strip Ensembl version suffix in 'gene' column ---
    if "gene" in df.columns:
        ens_pattern = r"(ens[gtp]\d+)\.\d+"
        df["gene"] = df["gene"].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df["gene"] = df["gene"].replace("nan", pd.NA)

    return df

In [77]:
geo_expr_clean = clean_geo_expr(geo_expr)
geo_expr_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_expr_clean.parquet", index=False)

In [78]:
geo_expr_clean

,gene,gsm101610,gsm101615,gsm101616,gsm101667,gsm101668,gsm101671,gsm101672,gsm101673,gsm101674,...,gsm960289,gsm960290,gsm960291,gsm960292,gsm960293,gsm960294,gsm960295,gsm960296,gsm960297,gsm960298
0,ensg00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ensg00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ensg00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952
3,ensg00000000457,58.934814,95.068222,94.900459,51.810070,52.327530,50.011375,53.793667,51.172344,52.677490,...,35.778347,57.736485,25.528204,24.593216,23.337006,47.793488,41.268002,64.042831,37.363773,61.027988
4,ensg00000000460,136.418900,257.929169,271.317230,162.090073,160.913849,206.788635,209.966019,134.272583,147.408554,...,14.693585,19.083935,114.056015,137.948593,149.641708,111.793892,98.536613,167.151016,12.268769,24.296593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19909,ensg00000288586,43.665115,48.006367,49.420658,48.256123,51.318356,44.370525,44.745148,46.820324,44.311581,...,8.703340,8.405022,9.103144,7.817888,8.262957,8.735815,8.062046,9.254727,9.124946,8.299891
19910,ensg00000288611,67.741661,56.979630,64.062752,56.845547,72.174866,58.450623,70.556740,61.306789,75.601913,...,15.711375,14.327072,17.170210,15.914798,15.235712,14.625729,13.727829,14.823915,16.405500,12.725769
19911,ensg00000288612,106.467583,66.520317,63.365860,74.388527,65.049774,79.616753,76.948509,79.405640,67.676636,...,9.806347,8.511088,11.726264,11.860399,11.062765,9.372013,9.360144,9.762713,9.330399,9.837292
19912,ensg00000288658,17.560717,126.690521,109.173874,35.579201,42.132763,15.810733,18.250271,17.695656,16.866858,...,3.866174,3.918684,4.088594,3.400458,3.349409,3.293435,3.202622,3.740396,3.607048,3.733616


## 4. Proteomics

In [79]:
def clean_proteomics(df):
    """
    Clean the proteomics table (proteomics).

    Structure: rows = samples (ach- IDs in 'unnamed: 0'),
    columns = protein headers like 'a0av96 (rbm47)' = uniprot_id (gene_symbol).
    Sometimes the gene symbol is absent: 'a0av96'.

    Steps:
        1. Lowercase column names + all string values
        2. Rename 'unnamed: 0' -> 'depmap_id'
        3. Reduce protein headers to the bare uniprot id, and build a
           separate mapping table {uniprot_id, gene_symbol, original_header}
           so no information is lost.

    Returns:
        (df_clean, protein_map):
            df_clean (pd.DataFrame): matrix with uniprot-id column headers
            protein_map (pd.DataFrame): uniprot_id | gene_symbol | original_header
    """
    df = df.copy()

    # --- Step 1: lowercase column names + strip ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 2: rename id column ---
    df = df.rename(columns={"unnamed: 0": "depmap_id"})

    # --- Step 1 (values): lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: parse 'uniprot (gene)' headers, build mapping, rename cols ---
    # matches: 'a0av96 (rbm47)'  -> id='a0av96', gene='rbm47'
    #          'a0av96'          -> id='a0av96', gene=None
    pattern = re.compile(r"^\s*([a-z0-9\-]+)\s*(?:\(([^)]*)\))?\s*$", re.IGNORECASE)

    id_cols = ["depmap_id"]  # columns to leave untouched
    map_rows = []
    new_names = {}

    for col in df.columns:
        if col in id_cols:
            continue
        m = pattern.match(str(col))
        if m:
            uniprot_id = m.group(1).strip().lower()
            gene       = (m.group(2).strip().lower() if m.group(2) else pd.NA)
        else:
            uniprot_id = str(col).strip().lower()
            gene       = pd.NA

        new_names[col] = uniprot_id
        map_rows.append({
            "uniprot_id": uniprot_id,
            "gene_symbol": gene,
            "original_header": col,
        })

    df = df.rename(columns=new_names)
    protein_map = pd.DataFrame(map_rows)

    return df, protein_map

In [80]:
proteomics_clean, protein_map = clean_proteomics(proteomics)

proteomics_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/proteomics_clean.parquet", index=False)
protein_map.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/protein_map.parquet", index=False)

In [81]:
proteomics_clean

,depmap_id,a0av96,a0avf1,a0avg3,a0avi4,a0avk6,a0avt1,a0jlt2,a0jnw5,a0mz66,...,q9y6n7-2,q9y4e1-4,q9y6q5-2,q9y6k9-2,q9y3y2-3,q9y4p1-2,q9y6i3-1,q9y5v3-2,q9y575-3,q9y2l9-2
0,ach-000849,0.358567,-0.172066,NaN,NaN,NaN,-0.496842,0.267691,0.045754,0.065342,...,0.255903,-0.081142,1.680544,0.001122,0.684397,0.397332,0.209930,-0.197547,NaN,-1.100381
1,ach-000441,-1.112410,0.339446,NaN,NaN,NaN,-0.395633,-0.262715,-0.414909,-0.396370,...,0.199431,-0.300508,-0.414740,0.178375,-0.387377,-0.008456,-0.482339,0.148685,NaN,0.484499
2,ach-000248,0.855575,-0.181171,NaN,NaN,NaN,-0.284720,-0.436849,0.555351,1.550787,...,-0.849680,0.096941,1.120568,-0.512328,0.102240,-0.204140,0.039421,-0.761104,NaN,-1.191209
3,ach-000684,0.061377,-0.341233,NaN,NaN,NaN,1.481536,-0.006424,0.185271,-0.125160,...,-0.230147,0.148388,-0.162396,0.279161,-0.057371,-0.331465,-0.014308,0.617058,NaN,0.409230
4,ach-000856,0.284258,-0.059558,NaN,NaN,NaN,-0.520511,0.680164,-0.281309,-0.691303,...,-0.024407,0.698199,-0.518353,-0.292139,0.498029,0.283140,-0.123974,-0.560028,NaN,-0.120029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,ach-000846,0.348655,-1.032205,NaN,NaN,NaN,-0.611541,0.473297,-0.523671,1.909424,...,-0.943454,0.274945,0.221292,0.467104,-0.862766,-1.432287,-0.037488,-0.531392,NaN,0.235568
371,ach-000265,-1.442707,-0.684201,NaN,NaN,NaN,-0.729435,0.763294,0.043234,0.659305,...,1.375092,0.813708,-0.336331,0.962541,0.030003,-0.364443,0.060838,1.521803,NaN,-0.144655
372,ach-000006,0.436273,-0.770205,NaN,NaN,NaN,0.965691,-0.135442,0.204895,-0.680679,...,-1.813197,-0.879452,-0.671954,0.506344,0.291667,1.241690,-0.264163,-0.065897,NaN,-0.145264
373,ach-000696,-1.585523,0.289067,NaN,NaN,NaN,0.373398,-0.329539,-0.114584,-0.249955,...,1.406823,0.685588,-0.813471,0.767487,-0.489870,-0.457437,0.373700,0.348390,NaN,0.671977


## 5. Fusions

In [82]:
def clean_fusions(df):
    """
    Clean the fusions table (fusions).

    Steps:
        1. Lowercase column names
        2. Lowercase all string values
        3. Rename 'unnamed: 0' -> 'fusion_index'
        4. Split 'gene1(ens id)'  -> 'gene1', 'gene1_ens_id'
        5. Split 'gene2(ens id)'  -> 'gene2', 'gene2_ens_id'
        6. Strip/collapse whitespace
        7. Strip Ensembl version suffix (.21/.0/.00) from the ens id parts
        8. Empty ()        -> ens id = NA (empty)
           Placeholder (.) -> kept as-is ('.', etc.)

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 3: rename id column ---
    df = df.rename(columns={"unnamed: 0": "fusion_index"})

    # --- Steps 2 & 6: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- helper to split 'gene (ens id)' into (gene, ens_id) ---
    # captures: group1 = text before '(', group2 = text inside '()'
    split_pat = re.compile(r"^(.*?)\s*\(([^)]*)\)\s*$")
    ens_version = re.compile(r"(ens[gtp]\d+)\.\d+", re.IGNORECASE)

    def split_gene_ens(val):
        if pd.isna(val):
            return (pd.NA, pd.NA)
        s = str(val).strip()
        m = split_pat.match(s)
        if m:
            gene = m.group(1).strip()
            ens  = m.group(2).strip()
            # Step 7: strip version suffix only on real ensembl IDs
            ens = ens_version.sub(r"\1", ens)
            # Step 8: empty () -> NA; placeholder like '.' kept as-is
            if ens == "":
                ens = pd.NA
            gene = gene if gene != "" else pd.NA
            return (gene, ens)
        # no parentheses found -> whole value is the gene, no ens id
        return (s if s != "" else pd.NA, pd.NA)

    # --- Steps 4, 5, 7, 8 ---
    for src, gene_col, ens_col in [
        ("gene1(ens id)", "gene1", "gene1_ens_id"),
        ("gene2(ens id)", "gene2", "gene2_ens_id"),
    ]:
        if src in df.columns:
            parsed = df[src].apply(split_gene_ens)
            df[gene_col] = parsed.apply(lambda x: x[0])
            df[ens_col]  = parsed.apply(lambda x: x[1])
            df = df.drop(columns=[src])

    return df

In [83]:
fusions_clean = clean_fusions(fusions)
fusions_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/fusions_clean.parquet", index=False)

In [84]:
fusions_clean

,fusion_index,sequencingid,modelid,isdefaultentryformodel,modelconditionid,isdefaultentryformc,canonicalfusionname,totalreadsinsample,totalreadssupportingfusion,ffpm,...,coverage1,coverage2,tags,retained_protein_domains,direction1,direction2,gene1,gene1_ens_id,gene2,gene2_ens_id
0,0,cds-010xbm,ach-001113,yes,mc-001113-k2lr,yes,dlg1--serpini1,47434547,203,4.279581,...,1454,76,.,.,upstream,upstream,dlg1,ensg00000075711,serpini1,.
1,1,cds-010xbm,ach-001113,yes,mc-001113-k2lr,yes,adam17--itgb1bp1,47434547,138,2.909272,...,883,724,.,.,upstream,downstream,adam17,ensg00000151694,itgb1bp1,ensg00000119185
2,2,cds-010xbm,ach-001113,yes,mc-001113-k2lr,yes,adam17--itgb1bp1,47434547,57,1.201656,...,883,754,.,.,upstream,downstream,adam17,ensg00000151694,itgb1bp1,ensg00000119185
3,3,cds-010xbm,ach-001113,yes,mc-001113-k2lr,yes,adam17--itgb1bp1,47434547,50,1.054084,...,41,724,.,.,upstream,downstream,adam17,ensg00000151694,itgb1bp1,ensg00000119185
4,4,cds-010xbm,ach-001113,yes,mc-001113-k2lr,yes,adam17--itgb1bp1,47434547,22,0.463797,...,883,680,.,.,upstream,downstream,adam17,ensg00000151694,itgb1bp1,ensg00000119185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184232,184232,cds-zx92js,ach-000052,yes,mc-000052-uuw6,yes,mid1--rp11-120d5.1,79663811,1,0.012553,...,17,9,.,.,upstream,downstream,rp11-120d5.1,ensg00000234129,mid1,ensg00000101871
184233,184233,cds-zx92js,ach-000052,yes,mc-000052-uuw6,yes,arhgef35-as1--rp4-798c17.7,79663811,1,0.012553,...,59,93,.,.,upstream,upstream,rp4-798c17.7,ensg00000284644,arhgef35-as1,ensg00000244198
184234,184234,cds-zx92js,ach-000052,yes,mc-000052-uuw6,yes,arhgef35-as1--rp4-798c17.7,79663811,0,0.000000,...,114,58,.,.,upstream,upstream,rp4-798c17.7,ensg00000284644,arhgef35-as1,ensg00000244198
184235,184235,cds-zx92js,ach-000052,yes,mc-000052-uuw6,yes,arhgef35-as1--rp4-798c17.7,79663811,0,0.000000,...,253,58,.,.,upstream,upstream,rp4-798c17.7,ensg00000284644,arhgef35-as1,ensg00000244198


## 6. Mutations

In [85]:
def clean_mutations(df):
    """
    Clean the mutations table (mutations).

    Structure: ~1.07M rows x 70 cols. Sample key = 'profileid' (PR-).
    Ensembl IDs live in 'ensemblgeneid' (and 'ensemblfeatureid').

    Steps:
        1. Lowercase all column names
        2. Strip/collapse whitespace in string values
        3. Strip Ensembl version suffix (.5/.00/.0) from ensembl ID columns
        4. Lowercase all string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 4: whitespace clean + lowercase on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: strip Ensembl version suffix on ensembl ID columns only ---
    ens_pattern = r"(ens[gtp]\d+)\.\d+"
    ens_cols = [c for c in ["ensemblgeneid", "ensemblfeatureid"] if c in df.columns]
    for col in ens_cols:
        df[col] = df[col].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [86]:
mutations_clean = clean_mutations(mutations)
mutations_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mutations_clean.parquet", index=False)

In [87]:
mutations_clean

,chrom,pos,ref,alt,af,dp,refcount,altcount,gt,ps,...,gwaspmid,gtexgene,proveanprediction,amclass,ampathogenicity,rescue,rescuereason,profileid,hotspot,entrezgeneid
0,chr1,818203,g,a,0.240,27,21,6,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-t8saqo,False,400728.0
1,chr1,851926,g,a,0.158,17,15,2,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-lhmbt6,False,643837.0
2,chr1,924510,gc,aa,0.412,35,21,14,0|1,924510.0,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-ukiczk,False,148398.0
3,chr1,924657,c,g,0.437,17,9,8,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-sxfiuq,False,148398.0
4,chr1,924750,c,t,0.625,19,7,12,0/1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-dneoiz,False,148398.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066864,chry,20768944,g,t,0.193,34,28,6,0/1,NaN,...,NaN,NaN,NaN,ambiguous,0.3748,False,NaN,pr-jiwarm,False,140032.0
1066865,chry,26400957,t,a,0.561,2,1,1,0|1,26400941.0,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-rgjfnl,False,NaN
1066866,chry,26414504,t,a,0.987,80,0,80,1|1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-jcsmgb,False,NaN
1066867,chry,26543783,g,a,0.978,45,0,45,1|1,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,pr-7n7fxy,False,NaN


## 7. Cellosaurus

In [88]:
def clean_cellosaurus(df):
    """
    Clean the cellosaurus table (cellosaurus).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4/5. Rename:
            'identifier (cell line name)' -> 'cellosaurus_cell_line_name'
            'accession (cvcl_xxxx)'       -> 'cellosaurus_accession'
        6. Normalise 'cellosaurus_cell_line_name' in place -> remove separators
           (e.g. '537-mel' -> '537mel') to match the stripped naming convention

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Steps 4 & 5: rename key columns ---
    rename_map = {
        "identifier (cell line name)": "cellosaurus_cell_line_name",
        "accession (cvcl_xxxx)": "cellosaurus_accession",
    }
    df = df.rename(columns=rename_map)

    # --- Step 6: normalise cell line name IN PLACE ---
    # remove separators (spaces, hyphens, dots, underscores): '537-mel' -> '537mel'
    if "cellosaurus_cell_line_name" in df.columns:
        df["cellosaurus_cell_line_name"] = (
            df["cellosaurus_cell_line_name"].astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"[\s\-\.\_]", "", regex=True)
        )
        df["cellosaurus_cell_line_name"] = (
            df["cellosaurus_cell_line_name"].replace({"nan": pd.NA, "": pd.NA})
        )

    return df

In [89]:
cellosaurus_clean = clean_cellosaurus(cellosaurus)
cellosaurus_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/cellosaurus_clean.parquet", index=False)

In [90]:
cellosaurus_clean

,cellosaurus_cell_line_name,cellosaurus_accession,secondary accession number(s),synonyms,cross-references,references identifiers,web pages,comments,str profile data,diseases,species of origin,hierarchy,originate from same individual,sex of cell,age of donor at sampling,category,date (entry history)
0,#132pc31sce8,cvcl_b0t9,NaN,z48-5mg-70,wikidata; q108819335,patent=ep0501779a1;,NaN,group: patented cell line. || registration: in...,NaN,NaN,ncbi_taxid=10090; ! mus musculus (mouse),cvcl_d145 ! hl-1 friendly myeloma-653,NaN,NaN,NaN,hybridoma,created: 23-09-21; last updated: 30-01-24; ver...
1,#132pl12scd1,cvcl_b0t8,NaN,z48-5mg-63,wikidata; q108819336,patent=ep0501779a1;,NaN,group: patented cell line. || registration: in...,NaN,NaN,ncbi_taxid=10090; ! mus musculus (mouse),cvcl_d145 ! hl-1 friendly myeloma-653,NaN,NaN,NaN,hybridoma,created: 23-09-21; last updated: 30-01-24; ver...
2,#15310ln,cvcl_e548,NaN,15310-ln; ter461; ter-461; ter 461; ter479; te...,dbmhc; 48439 || ecacc; 94050311 || ihw; ihw093...,NaN,http://pathology.ucla.edu/workfiles/360cx.pdf ...,part of: 12th international histocompatibility...,NaN,NaN,ncbi_taxid=9606; ! homo sapiens (human),NaN,NaN,female,age unspecified,transformed cell line,created: 22-10-12; last updated: 30-01-24; ver...
3,#1615,cvcl_ka96,NaN,NaN,rcb; rcb4635 || wikidata; q54422067,pubmed=25400923;,NaN,monoclonal antibody isotype: igm. || monoclona...,NaN,NaN,ncbi_taxid=10090; ! mus musculus (mouse) || nc...,cvcl_4032 ! p3x63ag8.653,NaN,NaN,NaN,hybridoma,created: 22-08-17; last updated: 21-03-23; ver...
4,#40a,cvcl_iw91,NaN,NaN,wikidata; q54422071,pubmed=28159921;,NaN,characteristics: established from parent cell ...,NaN,ncit; c21619; mouse mesothelioma,ncbi_taxid=10090; ! mus musculus (mouse),cvcl_iw90 ! 40,NaN,male,1-2m,cancer cell line,created: 15-05-17; last updated: 29-06-23; ver...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152226,zzusahi001a,cvcl_zb29,NaN,kcna5-v259i-ipsc,hpscreg; zzusahi001-a || skip; skip005861 || w...,pubmed=32721895;,NaN,from: zhengzhou university second affiliated h...,NaN,ncit; c50466; atrial fibrillation || ordo; orp...,ncbi_taxid=9606; ! homo sapiens (human),NaN,NaN,female,57y,induced pluripotent stem cell,created: 02-07-20; last updated: 29-06-23; ver...
152227,zzusahi002a,cvcl_zb30,NaN,NaN,hpscreg; zzusahi002-a || wikidata; q98136743,pubmed=32911326;,NaN,from: zhengzhou university second affiliated h...,NaN,NaN,ncbi_taxid=9606; ! homo sapiens (human),NaN,NaN,female,32y,induced pluripotent stem cell,created: 02-07-20; last updated: 29-06-23; ver...
152228,zzusahi003a,cvcl_a3zf,NaN,NaN,hpscreg; zzusahi003-a || wikidata; q105511894,pubmed=33450697;,NaN,from: zhengzhou university second affiliated h...,NaN,ncit; c34807; marfan syndrome || ordo; orphane...,ncbi_taxid=9606; ! homo sapiens (human),NaN,NaN,female,26y,induced pluripotent stem cell,created: 12-01-21; last updated: 29-06-23; ver...
152229,zzusahi004a,cvcl_c6u7,NaN,NaN,biosamples; samea111442306 || hpscreg; zzusahi...,pubmed=36395689;,NaN,from: zhengzhou university second affiliated h...,NaN,ncit; c192195; long qt syndrome 11 || ordo; or...,ncbi_taxid=9606; ! homo sapiens (human),NaN,NaN,male,3y,induced pluripotent stem cell,created: 21-03-23; last updated: 30-01-24; ver...


## 8. Depmap_Profiles

In [91]:
def clean_depmap_profiles(df):
    """
    Clean the depmap_profiles table (depmap_profiles).

    This is the bridge table linking PR- profile IDs to ACH- model IDs.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [92]:
depmap_profiles_clean = clean_depmap_profiles(depmap_profiles)
depmap_profiles_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_profiles_clean.parquet", index=False)

In [93]:
depmap_profiles_clean

,profileid,modelcondition,modelid,datatype,weskit
0,pr-00utu3,mc-001131-kkjv,ach-001131,wgs,NaN
1,pr-01r7om,mc-000957-yckn,ach-000957,rna,NaN
2,pr-02xmlg,mc-002785-qo9e,ach-002785,rna,NaN
3,pr-04vvbz,mc-001289-bpdi,ach-001289,wes,ice
4,pr-09gmei,mc-000520-yim7,ach-000520,rna,NaN
...,...,...,...,...,...
3825,pr-zym15a,mc-000796-mvbn,ach-000796,wes,agilent
3826,pr-zytjvt,mc-000278-uc8l,ach-000278,wgs,NaN
3827,pr-zzhtvc,mc-000231-teoc,ach-000231,rna,NaN
3828,pr-zzjqoa,mc-000310-adwh,ach-000310,wes,ice


## 9. sample_info

In [94]:
def clean_sample_info(df):
    """
    Clean the sample_info table (sample_info).

    Central cell-line dimension table. Key columns include depmap_id (ACH-),
    cell_line_name, ccle_name, cosmicid, rrid, etc.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. Normalise 'cell_line_name' in place -> remove separators
           (e.g. '537-mel' -> '537mel')
        5. Drop 'stripped_cell_line_name'

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: normalise 'cell_line_name' IN PLACE ---
    # remove separators (spaces, hyphens, dots, underscores): '537-mel' -> '537mel'
    if "cell_line_name" in df.columns:
        df["cell_line_name"] = (
            df["cell_line_name"].astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"[\s\-\.\_]", "", regex=True)
        )
        df["cell_line_name"] = df["cell_line_name"].replace({"nan": pd.NA, "": pd.NA})

    # --- Step 5: drop stripped_cell_line_name ---
    if "stripped_cell_line_name" in df.columns:
        df = df.drop(columns=["stripped_cell_line_name"])

    return df

In [95]:
sample_info_clean = clean_sample_info(sample_info)
sample_info_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/sample_info_clean.parquet", index=False)

In [96]:
sample_info_clean

,depmap_id,cell_line_name,ccle_name,alias,cosmicid,sex,source,rrid,wtsi_master_cell_id,sample_collection_site,...,lineage_sub_subtype,lineage_molecular_subtype,default_growth_pattern,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,cellosaurus_ncit_disease,cellosaurus_ncit_id,cellosaurus_issues
0,ach-000016,slr21,slr21_kidney,NaN,NaN,NaN,academic lab,cvcl_v607,NaN,kidney,...,NaN,NaN,NaN,NaN,NaN,pt-jnarlb,NaN,clear cell renal cell carcinoma,c4033,NaN
1,ach-000032,mhhcall3,mhhcall3_haematopoietic_and_lymphoid_tissue,NaN,NaN,female,dsmz,cvcl_0089,NaN,bone_marrow,...,b_cell,NaN,NaN,NaN,NaN,pt-p2koyi,NaN,childhood b acute lymphoblastic leukemia,c9140,NaN
2,ach-000033,ncih1819,ncih1819_lung,NaN,NaN,female,academic lab,cvcl_1497,NaN,lymph_node,...,nsclc_adenocarcinoma,NaN,NaN,NaN,NaN,pt-9p1wqv,NaN,lung adenocarcinoma,c3512,NaN
3,ach-000043,hs895t,hs895t_fibroblast,NaN,NaN,female,atcc,cvcl_0993,NaN,fibroblast,...,NaN,NaN,2d: adherent,NaN,NaN,pt-rtuvzq,NaN,melanoma,c3224,NaN
4,ach-000049,hekte,hekte_kidney,NaN,NaN,NaN,academic lab,cvcl_ws59,NaN,kidney,...,NaN,NaN,NaN,immortalized,NaN,pt-qwyygr,NaN,NaN,NaN,no information is available about this cell li...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1835,ach-002393,croap3,croap3_haematopoietic_and_lymphoid_tissue,NaN,NaN,male,sanger,cvcl_1810,NaN,ascites,...,b_cell_primary_effusion,NaN,NaN,NaN,NaN,pt-tc0lzm,NaN,primary effusion lymphoma,c6915,NaN
1836,ach-002394,geo,geo_large_intestine,NaN,NaN,NaN,sanger,cvcl_0271,NaN,large_intestine,...,NaN,NaN,NaN,NaN,NaN,pt-fa1q9q,NaN,colon carcinoma,c4910,NaN
1837,ach-002395,huh6clone5,huh6clone5_liver,NaN,NaN,male,sanger,cvcl_1296,NaN,liver,...,NaN,NaN,NaN,NaN,NaN,pt-ttixsl,ach-000671,hepatoblastoma,c3728,NaN
1838,ach-002396,sarc9371,sarc9371_bone,NaN,NaN,NaN,sanger,cvcl_5g89,NaN,bone,...,NaN,NaN,NaN,NaN,NaN,pt-715fdc,NaN,osteosarcoma,c9145,NaN


## 10. Geo Info

In [97]:
def clean_geo_info(df):
    """
    Clean the geo_info table (geo_info).

    GEO sample metadata. Key join column is geo_accession (GSM IDs).
    NOTE: this table had known leading/trailing whitespace on the cell line
    field causing silent join failures — the whitespace strip here fixes that.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. Normalise cell-line name columns in place -> remove separators
           (e.g. '537-mel' -> '537mel')   [applies to 'cell_lne' and 'cellline']

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: normalise cell-line name columns IN PLACE ---
    # both 'cell_lne' and 'cellline' hold cell line names
    cell_line_cols = ["cellline"]
    for col in cell_line_cols:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                .str.lower()
                .str.strip()
                .str.replace(r"[\s\-\.\_]", "", regex=True)
            )
            df[col] = df[col].replace({"nan": pd.NA, "": pd.NA})
        else:
            print(f"  ⚠ geo_info: expected cell-line column '{col}' not found.")

    return df

In [98]:
geo_info_clean = clean_geo_info(geo_info)
geo_info_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_info_clean.parquet", index=False)

In [99]:
geo_info_clean

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,cell_line_trimmed
0,gsm101610,gsm101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0131,a172,cello geo gsm,NaN
1,gsm101615,gsm101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,NaN
2,gsm101616,gsm101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,NaN
3,gsm101667,gsm101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,NaN
4,gsm101668,gsm101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262,gsm960294,gsm960294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3263,gsm960295,gsm960295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3264,gsm960296,gsm960296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3265,gsm960297,gsm960297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_7082,NaN,NaN,NaN


## 11. HPA Desc

In [100]:
def clean_hpa_desc(df):
    """
    Clean the hpa_desc table (hpa_desc).

    HPA cell line metadata. Key columns: 'cell line' (join to sample_info /
    cellosaurus), 'cellosaurus id', disease info.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. Normalise 'cell line' name in place -> remove separators
           (e.g. '537-mel' -> '537mel')

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: normalise 'cell line' name IN PLACE ---
    # remove separators (spaces, hyphens, dots, underscores): '537-mel' -> '537mel'
    if "cell line" in df.columns:
        df["cell line"] = (
            df["cell line"].astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"[\s\-\.\_]", "", regex=True)
        )
        df["cell line"] = df["cell line"].replace({"nan": pd.NA, "": pd.NA})

    return df

In [101]:
hpa_desc_clean = clean_hpa_desc(hpa_desc)
hpa_desc_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_desc_clean.parquet", index=False)

In [102]:
hpa_desc_clean

,cell line,disease,disease subtype,cellosaurus id,patient,primary/metastasis,sample collection site
0,143b,bone cancer,osteosarcoma,cvcl_2270,13,primary,bone
1,22rv1,prostate cancer,adenocarcinoma,cvcl_1045,male,primary,prostate
2,23132/87,gastric cancer,adenocarcinoma,cvcl_1046,"male, 72",primary,stomach
3,253j,bladder cancer,carcinoma,cvcl_7935,"male, 53",metastasis,lymph node
4,253jbv,bladder cancer,carcinoma,cvcl_7937,"male, 53",metastasis,lymph node
...,...,...,...,...,...,...,...
1201,yh13,brain cancer,glioblastoma,cvcl_1795,"male, 40",primary,central nervous system
1202,ykg1,brain cancer,glioblastoma,cvcl_1796,"female, 53",primary,central nervous system
1203,ymb1,breast cancer,breast ductal carcinoma,cvcl_2814,63,metastasis,ascites
1204,zr751,breast cancer,breast ductal carcinoma,cvcl_0588,"female, 63",metastasis,ascites


## 12. Metabolomics

In [103]:
def clean_metabolomics(df):
    """
    Clean the metabolomics table (metabolomics).

    Wide CCLE metabolomics. Keys: 'ccle_id' and 'depmap_id' (ACH-).
    Remaining ~225 columns are metabolite abundances (numeric).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. From 'ccle_id', add two derived columns:
             - 'ccle_id_tissue' : tissue suffix only (e.g. 'lung')
             - 'cell_line_name' : bare normalised name (e.g. 'dms53')

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: derive tissue + bare name from 'ccle_id' ---
    if "ccle_id" in df.columns:
        tissue_suffixes = [
            "haematopoietic_and_lymphoid_tissue", "central_nervous_system",
            "upper_aerodigestive_tract", "autonomic_ganglia", "matched_normal_tissue",
            "large_intestine", "biliary_tract", "salivary_gland", "soft_tissue",
            "urinary_tract", "endometrium", "oesophagus", "fibroblast", "pancreas",
            "prostate", "pleura", "stomach", "thyroid", "kidney", "breast", "ovary",
            "liver", "bone", "lung", "skin",
        ]
        tissue_suffixes = sorted(tissue_suffixes, key=len, reverse=True)
        suffix_pattern = r"_(" + "|".join(tissue_suffixes) + r")$"

        # extra col 1: the tissue suffix (captured), e.g. 'lung'
        df["ccle_id_tissue"] = (
            df["ccle_id"].astype(str).str.extract(suffix_pattern, expand=False)
        )

        # extra col 2: bare normalised name (suffix removed + separators stripped)
        df["cell_line_name"] = (
            df["ccle_id"].astype(str)
            .str.lower()
            .str.strip()
            .str.replace(suffix_pattern, "", regex=True)
            .str.replace(r"[\s\-\.\_]", "", regex=True)
        )
        df["cell_line_name"] = df["cell_line_name"].replace({"nan": pd.NA, "": pd.NA})

    return df

In [104]:
metabolomics_clean = clean_metabolomics(metabolomics)
metabolomics_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/metabolomics_clean.parquet", index=False)

In [105]:
metabolomics_clean

,ccle_id,depmap_id,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate,aconitate,adenine,adipate,alpha-ketoglutarate,...,c56:6 tag,c56:5 tag,c56:4 tag,c56:3 tag,c56:2 tag,c58:8 tag,c58:7 tag,c58:6 tag,ccle_id_tissue,cell_line_name
0,dms53_lung,ach-000698,6.112727,6.034198,5.896896,6.000532,5.513618,5.868529,5.977177,5.693074,...,6.091089,6.257711,6.372732,6.202511,5.939576,6.309821,6.115974,5.999436,lung,dms53
1,sw1116_large_intestine,ach-000489,5.577413,5.727045,5.111468,6.073250,5.802494,5.824473,5.888821,5.768379,...,6.378052,6.341043,6.360945,6.333540,6.137271,7.065858,6.832174,6.363064,large_intestine,sw1116
2,ncih1694_lung,ach-000431,5.886398,5.574881,5.541259,5.848375,5.665026,5.875548,5.894904,5.839640,...,5.837980,5.913350,6.137530,5.807546,5.704149,5.881193,5.785208,5.504225,lung,ncih1694
3,p3hr1_haematopoietic_and_lymphoid_tissue,ach-000707,5.770030,6.099229,6.233259,5.543495,5.767759,6.155905,6.111148,5.949481,...,6.282113,6.248667,6.109480,6.043570,5.846802,6.429402,5.779815,6.241530,haematopoietic_and_lymphoid_tissue,p3hr1
4,hut78_haematopoietic_and_lymphoid_tissue,ach-000509,5.480683,5.469742,6.509397,6.251005,5.190578,5.897085,6.148333,5.607481,...,6.676390,6.695659,6.751029,6.385056,6.682612,6.757899,6.728570,6.879260,haematopoietic_and_lymphoid_tissue,hut78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,sf268_central_nervous_system,ach-000655,5.977636,6.026483,6.480536,6.112056,5.878324,5.732508,5.981183,6.459139,...,5.581725,5.506887,5.362247,5.322091,5.556126,5.544802,4.897559,5.506999,central_nervous_system,sf268
924,sf539_central_nervous_system,ach-000273,5.957233,6.090834,5.323475,6.145795,5.741201,5.605435,6.184192,6.004195,...,5.720680,5.387960,5.275798,5.289684,5.494241,5.533795,5.139488,5.452544,central_nervous_system,sf539
925,snb75_central_nervous_system,ach-000504,5.967707,5.931487,5.620542,5.955261,5.935244,5.384674,5.955886,6.239657,...,5.630315,5.433324,5.330957,5.481404,5.570553,5.146598,5.010639,5.114385,central_nervous_system,snb75
926,hop92_lung,ach-000825,5.962415,5.992640,6.296222,5.916386,6.043984,5.883976,5.954731,6.181054,...,6.248746,6.130000,5.873297,5.826041,5.997694,6.411166,6.105394,6.387275,lung,hop92


## 13. Mirna

In [106]:
def clean_mirna(df):
    """
    Clean the mirna table (mirna).

    Structure: 'name' (miRNA id, e.g. hsa-mir-21), 'description',
    then ~954 cell-line columns CCLE-style (e.g. 'dms53_lung').

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. In 'name', strip trailing decimal suffix (.0 / .00 / .000)

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: strip trailing decimal suffix in 'name' ---
    if "name" in df.columns:
        df["name"] = df["name"].astype(str).str.replace(r"\.0+$", "", regex=True)
        df["name"] = df["name"].replace("nan", pd.NA)

    return df

In [107]:
mirna_clean = clean_mirna(mirna)
mirna_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mirna_clean.parquet", index=False)


In [108]:
mirna

,Name,Description,DMS53_LUNG,SW1116_LARGE_INTESTINE,NCIH1694_LUNG,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,UMUC3_URINARY_TRACT,HOS_BONE,HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,...,MOLT3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,HOP62_LUNG,EKVX_LUNG,OVCAR5_OVARY,UO31_KIDNEY,SF268_CENTRAL_NERVOUS_SYSTEM,SF539_CENTRAL_NERVOUS_SYSTEM,SNB75_CENTRAL_NERVOUS_SYSTEM,HOP92_LUNG,MUTZ3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
0,nmiR00001.1,hsa-let-7a,4362.58,5191.50,24991.05,3253.83,225.28,35051.17,5706.63,1821.46,...,30327.89,32129.83,18651.43,17551.39,30510.57,23953.54,26114.02,14115.97,13986.26,244.46
1,nmiR00002.1,hsa-let-7b,187.44,868.22,5066.09,74.21,35.74,1014.65,1211.27,22.54,...,2618.06,7962.03,3990.50,2629.28,5482.37,2718.48,2094.48,1318.81,1893.46,98.57
2,nmiR00003.1,hsa-let-7c,267.03,244.88,818.49,53.28,27.19,473.15,163.82,437.44,...,431.15,1573.10,528.74,506.52,1019.14,491.97,424.39,1830.06,355.16,77.54
3,nmiR00004.1,hsa-let-7d,868.11,556.55,3661.86,315.87,94.00,1766.54,904.84,133.18,...,8721.99,6134.93,6059.74,4448.79,4296.90,4183.58,6098.12,3645.09,970.48,197.15
4,nmiR00005.1,hsa-let-7e,1.04,92.02,942.62,1.90,0.77,279.63,168.07,1.02,...,99.07,2184.77,3893.23,806.47,1455.61,508.19,1872.56,1188.79,685.53,1.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,nmiR00796.1,kshv-miR-K12-7,13.61,34.14,17.45,15.23,15.54,19.52,16.98,17.42,...,21.09,11.08,21.62,21.48,24.37,23.43,33.17,17.69,27.87,21.03
730,nmiR00797.1,kshv-miR-K12-8,4.19,13.36,5.81,5.71,8.54,3.55,4.24,4.10,...,4.59,0.80,4.16,3.31,8.48,3.60,3.43,2.65,6.19,9.20
731,nmiR00798.1,kshv-miR-K12-9,27.23,59.37,44.60,34.25,17.87,31.96,26.32,47.12,...,64.21,14.25,44.07,39.66,39.20,23.43,35.46,30.08,35.11,48.63
732,nmiR00799.1,mcv-miR-M1-3p,9.42,13.36,7.76,7.61,9.33,10.65,11.88,11.27,...,5.50,4.75,4.99,12.39,10.59,5.40,12.58,11.50,3.09,3.94


## 14. Signatures

In [109]:
def clean_signatures(df):
    """
    Clean the signatures table (signatures).

    Pre-computed global omics signature scores. Connects via modelid (ACH-)
    and sequencingid (PR-).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. Rename 'unnamed: 0' -> 'signature_index'

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 4: rename id column ---
    df = df.rename(columns={"unnamed: 0": "signature_index"})

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [110]:
signatures_clean = clean_signatures(signatures)
signatures_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/signatures_clean.parquet", index=False)

In [111]:
signatures

,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
0,0,CDS-00Nrci,ACH-000839,MC-000839-krru,Yes,Yes,3.68,0.107443,1.0,0.502634,3.158291,20.0
1,1,CDS-051xn7,ACH-000041,MC-000041-uPBf,Yes,Yes,2.21,0.130089,1.0,0.523865,3.236089,19.0
2,2,CDS-099jzP,ACH-002046,MC-002046-oaX8,Yes,Yes,2.87,0.222342,1.0,0.679772,3.326715,30.0
3,3,CDS-0A4mDu,ACH-002048,MC-002048-52d6,Yes,Yes,2.48,NaN,NaN,NaN,NaN,NaN
4,4,CDS-0Eax8o,ACH-000042,MC-000042-eOnX,Yes,Yes,2.07,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3016,3016,CDS-zuxWuZ,ACH-001622,MC-001622-QSPg,No,No,4.01,NaN,NaN,NaN,NaN,NaN
3017,3017,CDS-zvEfPE,ACH-001379,MC-001379-q3xU,Yes,Yes,3.78,0.180697,0.0,0.572390,2.586749,24.0
3018,3018,CDS-zvvAOM,ACH-001690,MC-001690-yWll,Yes,Yes,2.84,0.171637,1.0,0.510565,3.479860,22.0
3019,3019,CDS-zyxdAN,ACH-002225,MC-002225-0ukX,Yes,Yes,0.77,0.237493,1.0,0.726200,3.307654,22.0


## Saving Clean Paraquets and Table Summary in Excel File

In [112]:
import pandas as pd

# ---------------------------------------------------------------
# LOAD ALL CLEAN TABLES
# ---------------------------------------------------------------
table_clean = {
    "hpa_rna_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_rna_clean.parquet"),
    "depmap_expr_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_expr_clean.parquet"),
    "geo_expr_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_expr_clean.parquet"),
    "proteomics_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/proteomics_clean.parquet"),
    "protein_map_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/protein_map.parquet"),
    "fusions_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/fusions_clean.parquet"),
    "mutations_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mutations_clean.parquet"),
    "cellosaurus_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/cellosaurus_clean.parquet"),
    "depmap_profiles_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_profiles_clean.parquet"),
    "sample_info_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/sample_info_clean.parquet"),
    "geo_info_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_info_clean.parquet"),
    "hpa_desc_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_desc_clean.parquet"),
    "metabolomics_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/metabolomics_clean.parquet"),
    "mirna_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mirna_clean.parquet"),
    "signatures_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/signatures_clean.parquet")
}

# ---------------------------------------------------------------
# BUILD COLUMN-LEVEL SUMMARY
# ---------------------------------------------------------------
def build_table_summary(df, n_samples=3):
    """
    Per-column summary:
    column | dtype | sample_values | missing_% | unique_values
           | primary_key | duplicates
    """
    rows = []
    n_rows = len(df)

    for col in df.columns:
        s = df[col]

        n_missing = s.isna().sum()
        n_present = n_rows - n_missing
        n_unique = s.nunique(dropna=True)
        n_duplicates = n_present - n_unique
        missing_pct = round((n_missing / n_rows) * 100, 2) if n_rows else 0

        is_pk = (n_missing == 0) and (n_duplicates == 0) and (n_rows > 0)

        sample = s.dropna().astype(str).head(n_samples).tolist()

        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "sample_values": ", ".join(sample),
            "missing_%": missing_pct,
            "unique_values": n_unique,
            "primary_key": is_pk,
            "duplicates": n_duplicates
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# EXPORT TO EXCEL (ONE SHEET PER TABLE)
# ---------------------------------------------------------------
def export_clean_report(tables: dict, out_path, n_samples=3):

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for name, df in tables.items():

            summary = build_table_summary(df, n_samples=n_samples)
            sheet_name = name[:31]  # Excel limit

            header = pd.DataFrame({
                "info": [
                    f"TABLE: {name}",
                    f"shape: {df.shape[0]} rows x {df.shape[1]} cols"
                ]
            })

            header.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                header=False,
                startrow=0
            )

            summary.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                startrow=3
            )

    print(f"Written {len(tables)} sheets to: {out_path}")


# ---------------------------------------------------------------
# RUN REPORT
# ---------------------------------------------------------------
OUT = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/clean_data_report.xlsx"

export_clean_report(table_clean, OUT, n_samples=3)

Written 15 sheets to: /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/clean_data_report.xlsx


## Checking Data Overlap in every Table

In [113]:
"""
=====================================================================
MASTER GENE LIST — distinct ENSG IDs from AUTHORITATIVE GENE COLUMNS only
(no free-text fields like cellosaurus comments / mutation annotations)
=====================================================================
"""
import pandas as pd
import re

# Where the ENSG legitimately lives in each gene-bearing table.
ENSG_ID_COLUMNS = {
    "hpa_rna_clean":     ["gene"],
    "geo_expr_clean":    ["gene"],
    "mutations_clean":   ["ensemblgeneid"],
    "fusions_clean":     ["gene1_ens_id", "gene2_ens_id"],
    "depmap_expr_clean": ["ensg_id"],     # genes in a row column (transposed form)
    # all other tables have no gene id -> not listed -> skipped
}


def extract_ensg_set(series):
    """Return a SET of distinct lowercase ensg ids from a column's values."""
    found = (
        series.dropna()
        .astype(str)
        .str.extract(r"(ens[gtp]\d+)", flags=re.IGNORECASE)[0]
        .dropna()
        .str.lower()
    )
    return set(found.unique())


def build_master_gene_list(tables: dict, verbose=True):
    """
    Scan ONLY designated gene id columns. Returns de-duplicated set
    + per-table breakdown.
    """
    all_ensg = set()
    breakdown = []

    for name, df in tables.items():
        table_ids = set()
        for col in ENSG_ID_COLUMNS.get(name, []):
            if col in df.columns:
                table_ids |= extract_ensg_set(df[col])
            elif verbose:
                print(f"  ⚠ {name}: expected column '{col}' not found. "
                      f"cols: {[c for c in df.columns if 'ens' in str(c).lower() or 'gene' in str(c).lower()][:5]}")

        all_ensg |= table_ids
        breakdown.append({"table": name, "n_ensg_found": len(table_ids)})
        if verbose:
            print(f"{name:24s}: {len(table_ids):>6,} distinct ENSG")

    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_ensg_found", ascending=False
    ).reset_index(drop=True)
    return all_ensg, breakdown_df


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
master_ensg, breakdown = build_master_gene_list(table_clean)

print("\n" + "=" * 50)
print(f"MASTER GENE LIST: {len(master_ensg):,} distinct ENSG IDs total")
print("=" * 50)
print("\nPer-table breakdown:")
print(breakdown.to_string(index=False))

master_gene_df = pd.DataFrame(
    sorted(master_ensg), columns=["master_ensembl_gene_id"]
)
print("\nSample of master gene list:")
print(master_gene_df.head())

hpa_rna_clean           : 20,162 distinct ENSG
depmap_expr_clean       : 53,961 distinct ENSG
geo_expr_clean          : 19,914 distinct ENSG
proteomics_clean        :      0 distinct ENSG
protein_map_clean       :      0 distinct ENSG
fusions_clean           : 25,356 distinct ENSG
mutations_clean         : 19,798 distinct ENSG
cellosaurus_clean       :      0 distinct ENSG
depmap_profiles_clean   :      0 distinct ENSG
sample_info_clean       :      0 distinct ENSG
geo_info_clean          :      0 distinct ENSG
hpa_desc_clean          :      0 distinct ENSG
metabolomics_clean      :      0 distinct ENSG
mirna_clean             :      0 distinct ENSG
signatures_clean        :      0 distinct ENSG

MASTER GENE LIST: 54,426 distinct ENSG IDs total

Per-table breakdown:
                table  n_ensg_found
    depmap_expr_clean         53961
        fusions_clean         25356
        hpa_rna_clean         20162
       geo_expr_clean         19914
      mutations_clean         19798
     pr

In [114]:
"""
=====================================================================
ADD master_ensembl_gene_id COLUMN TO ALL TABLES
- Tables with ENSG in a row column -> fill from that column
- fusions                          -> gene1_ens_id, fallback gene2_ens_id
- Tables with no gene id           -> empty (NA) column
=====================================================================
"""
import pandas as pd
import re

# Map: table -> source column holding the ENSG (None = no gene -> empty col)
GENE_SOURCE = {
    "hpa_rna_clean":         "gene",
    "geo_expr_clean":        "gene",
    "mutations_clean":       "ensemblgeneid",
    "fusions_clean":         "gene1_ens_id",   # fallback to gene2_ens_id below
    "depmap_expr_clean":     "ensg_id",        # transposed form: gene in row column
    "proteomics_clean":      None,             # protein headers, no gene row col
    "cellosaurus_clean":     None,
    "depmap_profiles_clean": None,
    "sample_info_clean":     None,
    "geo_info_clean":        None,
    "hpa_desc_clean":        None,
    "metabolomics_clean":    None,
    "mirna_clean":           None,
    "signatures_clean":      None,
}

FUSIONS_FALLBACK_COL = "gene2_ens_id"


def extract_ensg(series):
    """Extract bare lowercase ENSG from a column's values; NA where none."""
    return (
        series.astype(str)
        .str.extract(r"(ens[gtp]\d+)", flags=re.IGNORECASE)[0]
        .str.lower()
    )


def add_master_gene_id(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = GENE_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_ensembl_gene_id"] = extract_ensg(df[source_col])

            # fusions: fall back to gene2 where gene1 yields no ENSG
            if name == "fusions_clean" and FUSIONS_FALLBACK_COL in df.columns:
                g2 = extract_ensg(df[FUSIONS_FALLBACK_COL])
                df["master_ensembl_gene_id"] = df["master_ensembl_gene_id"].fillna(g2)
        else:
            df["master_ensembl_gene_id"] = pd.NA
            if source_col is not None:
                gene_like = [c for c in df.columns
                             if "gene" in str(c).lower() or "ens" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"gene-like cols present: {gene_like}")

        updated[name] = df
        n_filled = df["master_ensembl_gene_id"].notna().sum()
        print(f"{name:24s}: master_ensembl_gene_id filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
tables = add_master_gene_id(table_clean)

# assign by key (robust to extra entries in the dict)
hpa_rna_clean         = tables["hpa_rna_clean"]
depmap_expr_clean     = tables["depmap_expr_clean"]
geo_expr_clean        = tables["geo_expr_clean"]
proteomics_clean      = tables["proteomics_clean"]
fusions_clean         = tables["fusions_clean"]
mutations_clean       = tables["mutations_clean"]
cellosaurus_clean     = tables["cellosaurus_clean"]
depmap_profiles_clean = tables["depmap_profiles_clean"]
sample_info_clean     = tables["sample_info_clean"]
geo_info_clean        = tables["geo_info_clean"]
hpa_desc_clean        = tables["hpa_desc_clean"]
metabolomics_clean    = tables["metabolomics_clean"]
mirna_clean           = tables["mirna_clean"]
signatures_clean      = tables["signatures_clean"]


# ---------------------------------------------------------------
# VERIFY against master_gene_df
# ---------------------------------------------------------------
print("\n" + "=" * 55)
print("VERIFICATION")
print("=" * 55)
print("All tables have master_ensembl_gene_id column:",
      all("master_ensembl_gene_id" in df.columns for df in tables.values()))

master_set = set(master_gene_df["master_ensembl_gene_id"].astype(str).str.lower())

print("\nGene IDs in tables but NOT in master_gene_df:")
any_missing = False
for name, df in tables.items():
    found = set(df["master_ensembl_gene_id"].dropna().astype(str))
    missing = found - master_set
    if missing:
        any_missing = True
        print(f"  {name:24s}: {len(missing):>5,} not in master "
              f"(e.g. {list(missing)[:3]})")
if not any_missing:
    print("  none — every table's genes are covered by master_gene_df ✓")


hpa_rna_clean           : master_ensembl_gene_id filled in 24,315,372 of 24,315,372 rows
depmap_expr_clean       : master_ensembl_gene_id filled in    53,961 of    53,961 rows
geo_expr_clean          : master_ensembl_gene_id filled in    19,914 of    19,914 rows
proteomics_clean        : master_ensembl_gene_id filled in         0 of       375 rows
protein_map_clean       : master_ensembl_gene_id filled in         0 of    12,558 rows
fusions_clean           : master_ensembl_gene_id filled in   184,084 of   184,237 rows
mutations_clean         : master_ensembl_gene_id filled in 1,066,869 of 1,066,869 rows
cellosaurus_clean       : master_ensembl_gene_id filled in         0 of   152,231 rows
depmap_profiles_clean   : master_ensembl_gene_id filled in         0 of     3,830 rows
sample_info_clean       : master_ensembl_gene_id filled in         0 of     1,840 rows
geo_info_clean          : master_ensembl_gene_id filled in         0 of     3,267 rows
hpa_desc_clean          : master_ensembl_

In [115]:
"""
=====================================================================
MASTER CELL LINE LIST — distinct ACH- IDs from AUTHORITATIVE ID COLUMNS only
(excludes free-text fields like cellosaurus comments/cross-references/
synonyms/hierarchy and sample_info ccle_name/parent_depmap_id)
=====================================================================
"""
import pandas as pd
import re

# Only the proper ACH- key columns — one or more per table
ACH_ID_COLUMNS = {
    "sample_info_clean":     ["depmap_id"],
    "depmap_profiles_clean": ["modelid"],
    "fusions_clean":         ["modelid"],
    "signatures_clean":      ["modelid"],
    "metabolomics_clean":    ["depmap_id"],
    "proteomics_clean":      ["depmap_id"]
}


def extract_ach_set(series):
    """Return a SET of distinct lowercase ach- ids from a column."""
    found = (
        series.dropna()
        .astype(str)
        .str.extract(r"(ach-\d+)", flags=re.IGNORECASE)[0]
        .dropna()
        .str.lower()
    )
    return set(found.unique())


def build_master_cellline_list(tables: dict, verbose=True):
    """
    Scan ONLY designated ACH- id columns. Returns de-duplicated set
    + per-table breakdown.
    """
    all_ach = set()
    breakdown = []

    for name, df in tables.items():
        table_ids = set()
        for col in ACH_ID_COLUMNS.get(name, []):
            if col in df.columns:
                table_ids |= extract_ach_set(df[col])

        all_ach |= table_ids
        breakdown.append({"table": name, "n_ach_found": len(table_ids)})
        if verbose:
            print(f"{name:24s}: {len(table_ids):>6,} distinct ACH-")

    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_ach_found", ascending=False
    ).reset_index(drop=True)
    return all_ach, breakdown_df


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
master_ach, ach_breakdown = build_master_cellline_list(tables)

print("\n" + "=" * 50)
print(f"MASTER CELL LINE LIST: {len(master_ach):,} distinct ACH- IDs total")
print("=" * 50)
print("\nPer-table breakdown:")
print(ach_breakdown.to_string(index=False))

master_cellline_df = pd.DataFrame(
    sorted(master_ach), columns=["master_ach_id"]
)
print("\nSample of master cell line list:")
print(master_cellline_df.head())

hpa_rna_clean           :      0 distinct ACH-
depmap_expr_clean       :      0 distinct ACH-
geo_expr_clean          :      0 distinct ACH-
proteomics_clean        :    375 distinct ACH-
protein_map_clean       :      0 distinct ACH-
fusions_clean           :  1,699 distinct ACH-
mutations_clean         :      0 distinct ACH-
cellosaurus_clean       :      0 distinct ACH-
depmap_profiles_clean   :  1,822 distinct ACH-
sample_info_clean       :  1,840 distinct ACH-
geo_info_clean          :      0 distinct ACH-
hpa_desc_clean          :      0 distinct ACH-
metabolomics_clean      :    927 distinct ACH-
mirna_clean             :      0 distinct ACH-
signatures_clean        :  1,955 distinct ACH-

MASTER CELL LINE LIST: 2,127 distinct ACH- IDs total

Per-table breakdown:
                table  n_ach_found
     signatures_clean         1955
    sample_info_clean         1840
depmap_profiles_clean         1822
        fusions_clean         1699
   metabolomics_clean          927
     prot

In [116]:
"""
=====================================================================
ADD master_ach_id COLUMN TO ALL TABLES
- Tables with ACH- in a row value          -> fill from that column
- Tables with no ACH- (PR-/GSM/name keyed) -> empty (NA) column
  (these are reached later via bridges: depmap_profiles, geo_info, sample_info)
=====================================================================
"""

import pandas as pd
import re

# ---------------------------------------------------------------
# Map: table -> source column holding the ACH- id (None = empty col)
# ---------------------------------------------------------------
ACH_SOURCE = {
    "sample_info_clean":     "depmap_id",       # the ACH- registry hub
    "depmap_profiles_clean": "modelid",         # PR- -> ACH- bridge (ACH- side)
    "fusions_clean":         "modelid",         # ACH- keyed
    "signatures_clean":      "modelid",         # ACH- keyed
    "metabolomics_clean":    "depmap_id",       # ACH- keyed
    "proteomics_clean":      "depmap_id",       # ACH- keyed (renamed from unnamed: 0)
    # --- tables with NO direct ACH- (different ID system) -> empty ---
    "depmap_expr_clean":     None,              # PR- keyed (bridge later)
    "mutations_clean":       None,              # PR- keyed (bridge later)
    "geo_expr_clean":        None,              # GSM keyed
    "geo_info_clean":        None,              # GSM keyed
    "hpa_rna_clean":         None,              # cell-line-name keyed
    "hpa_desc_clean":        None,              # cell-line-name keyed
    "mirna_clean":           None,              # CCLE-name keyed
    "cellosaurus_clean":     None,              # CVCL- keyed
}


# ---------------------------------------------------------------
# STEP 1: extraction helper
# ---------------------------------------------------------------
def extract_ach(series):
    """Extract bare lowercase ACH- id from a column's values; NA where none."""
    return (
        series.astype(str)
        .str.extract(r"(ach-\d+)", flags=re.IGNORECASE)[0]
        .str.lower()
    )


# ---------------------------------------------------------------
# STEP 2: add master_ach_id to every table
# ---------------------------------------------------------------
def add_master_ach_id(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = ACH_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_ach_id"] = extract_ach(df[source_col])
        else:
            df["master_ach_id"] = pd.NA
            # warn if we EXPECTED a column but it isn't there
            if source_col is not None:
                ach_like = [c for c in df.columns
                            if "ach" in str(c).lower()
                            or "model" in str(c).lower()
                            or "depmap" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"ach-like cols present: {ach_like}")

        updated[name] = df
        n_filled = df["master_ach_id"].notna().sum()
        print(f"{name:24s}: master_ach_id filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
tables = add_master_ach_id(tables)

# assign by key (robust to extra entries in the dict)
hpa_rna_clean         = tables["hpa_rna_clean"]
depmap_expr_clean     = tables["depmap_expr_clean"]
geo_expr_clean        = tables["geo_expr_clean"]
proteomics_clean      = tables["proteomics_clean"]
fusions_clean         = tables["fusions_clean"]
mutations_clean       = tables["mutations_clean"]
cellosaurus_clean     = tables["cellosaurus_clean"]
depmap_profiles_clean = tables["depmap_profiles_clean"]
sample_info_clean     = tables["sample_info_clean"]
geo_info_clean        = tables["geo_info_clean"]
hpa_desc_clean        = tables["hpa_desc_clean"]
metabolomics_clean    = tables["metabolomics_clean"]
mirna_clean           = tables["mirna_clean"]
signatures_clean      = tables["signatures_clean"]


# ---------------------------------------------------------------
# STEP 3: verify against master_cellline_df
# ---------------------------------------------------------------
print("\n" + "=" * 55)
print("VERIFICATION")
print("=" * 55)

print("All tables have master_ach_id column:",
      all("master_ach_id" in df.columns for df in tables.values()))

master_ach_set = set(master_cellline_df["master_ach_id"].astype(str).str.lower()) \
    if "master_ach_id" in master_cellline_df.columns \
    else set(master_cellline_df.iloc[:, 0].astype(str).str.lower())

print("\nACH- IDs found in tables but NOT in master_cellline_df:")
any_missing = False
for name, df in tables.items():
    found = set(df["master_ach_id"].dropna().astype(str))
    missing = found - master_ach_set
    if missing:
        any_missing = True
        print(f"  {name:24s}: {len(missing):>5,} not in master "
              f"(e.g. {list(missing)[:3]})")
if not any_missing:
    print("  none — every table's ACH- IDs are covered by master_cellline_df ✓")

hpa_rna_clean           : master_ach_id filled in         0 of 24,315,372 rows
depmap_expr_clean       : master_ach_id filled in         0 of    53,961 rows
geo_expr_clean          : master_ach_id filled in         0 of    19,914 rows
proteomics_clean        : master_ach_id filled in       375 of       375 rows
protein_map_clean       : master_ach_id filled in         0 of    12,558 rows
fusions_clean           : master_ach_id filled in   184,237 of   184,237 rows
mutations_clean         : master_ach_id filled in         0 of 1,066,869 rows
cellosaurus_clean       : master_ach_id filled in         0 of   152,231 rows
depmap_profiles_clean   : master_ach_id filled in     3,830 of     3,830 rows
sample_info_clean       : master_ach_id filled in     1,840 of     1,840 rows
geo_info_clean          : master_ach_id filled in         0 of     3,267 rows
hpa_desc_clean          : master_ach_id filled in         0 of     1,206 rows
metabolomics_clean      : master_ach_id filled in       927 of 

In [117]:
print("=" * 60)
print("FINAL STATE — both spine columns on every table")
print("=" * 60)
print(f"{'table':<24} {'has_gene_id':<12} {'has_ach_id':<11} "
      f"{'gene_filled':>12} {'ach_filled':>12}")
print("-" * 75)

for name, df in tables.items():
    has_gene = "master_ensembl_gene_id" in df.columns
    has_ach  = "master_ach_id" in df.columns
    gene_filled = df["master_ensembl_gene_id"].notna().sum() if has_gene else 0
    ach_filled  = df["master_ach_id"].notna().sum() if has_ach else 0
    print(f"{name:<24} {str(has_gene):<12} {str(has_ach):<11} "
          f"{gene_filled:>12,} {ach_filled:>12,}")

# overall flags
all_have_gene = all("master_ensembl_gene_id" in df.columns for df in tables.values())
all_have_ach  = all("master_ach_id" in df.columns for df in tables.values())
print("\nAll tables have master_ensembl_gene_id:", all_have_gene)
print("All tables have master_ach_id:          ", all_have_ach)

FINAL STATE — both spine columns on every table
table                    has_gene_id  has_ach_id   gene_filled   ach_filled
---------------------------------------------------------------------------
hpa_rna_clean            True         True          24,315,372            0
depmap_expr_clean        True         True              53,961            0
geo_expr_clean           True         True              19,914            0
proteomics_clean         True         True                   0          375
protein_map_clean        True         True                   0            0
fusions_clean            True         True             184,084      184,237
mutations_clean          True         True           1,066,869            0
cellosaurus_clean        True         True                   0            0
depmap_profiles_clean    True         True                   0        3,830
sample_info_clean        True         True                   0        1,840
geo_info_clean           True         Tr

In [118]:
hpa_rna_clean

,gene,gene name,cell line,tpm,ptpm,ntpm,master_ensembl_gene_id,master_ach_id
0,ensg00000000003,tspan6,143b,22.0,27.6,25.9,ensg00000000003,<NA>
1,ensg00000000003,tspan6,22rv1,2.8,3.6,2.7,ensg00000000003,<NA>
2,ensg00000000003,tspan6,23132/87,6.2,7.5,7.5,ensg00000000003,<NA>
3,ensg00000000003,tspan6,253j,14.2,18.7,25.4,ensg00000000003,<NA>
4,ensg00000000003,tspan6,253jbv,13.0,17.1,18.5,ensg00000000003,<NA>
...,...,...,...,...,...,...,...,...
24315367,ensg00000291317,tmem276,yh13,16.8,21.8,15.6,ensg00000291317,<NA>
24315368,ensg00000291317,tmem276,ykg1,25.1,32.2,24.0,ensg00000291317,<NA>
24315369,ensg00000291317,tmem276,ymb1,42.4,55.9,47.8,ensg00000291317,<NA>
24315370,ensg00000291317,tmem276,zr751,30.0,40.9,30.8,ensg00000291317,<NA>


In [119]:
"""
=====================================================================
MASTER CVCL LIST — distinct CVCL- IDs from AUTHORITATIVE ID COLUMNS only
(Cellosaurus accessions; excludes free-text cross-references/comments/
synonyms/hierarchy where CVCL- mentions also appear)
=====================================================================
"""
import pandas as pd
import re

# Only the proper CVCL- key columns per table
CVCL_ID_COLUMNS = {
    "cellosaurus_clean": ["cellosaurus_accession"],   # the CVCL- registry (primary)
    "sample_info_clean": ["rrid"],          # sample_info's CVCL- xref col
    "hpa_desc_clean":    ["cellosaurus id"],
    "geo_info_clean":    ["cellosaurus_id"],
                        # HPA's CVCL- link col
}


def extract_cvcl_set(series):
    """Return a SET of distinct lowercase cvcl- ids from a column."""
    found = (
        series.dropna()
        .astype(str)
        .str.extract(r"(cvcl_[a-z0-9]+)", flags=re.IGNORECASE)[0]
        .dropna()
        .str.lower()
    )
    return set(found.unique())


def build_master_cvcl_list(tables: dict, verbose=True):
    """
    Scan ONLY designated CVCL- id columns. Returns de-duplicated set
    + per-table breakdown.
    """
    all_cvcl = set()
    breakdown = []

    for name, df in tables.items():
        table_ids = set()
        for col in CVCL_ID_COLUMNS.get(name, []):
            if col in df.columns:
                table_ids |= extract_cvcl_set(df[col])
            elif verbose:
                # column name may differ — show candidates
                cands = [c for c in df.columns if "cvcl" in str(c).lower()
                         or "cellosaurus" in str(c).lower()]
                print(f"  ⚠ {name}: '{col}' not found. candidates: {cands}")

        all_cvcl |= table_ids
        breakdown.append({"table": name, "n_cvcl_found": len(table_ids)})
        if verbose:
            print(f"{name:24s}: {len(table_ids):>6,} distinct CVCL-")

    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_cvcl_found", ascending=False
    ).reset_index(drop=True)
    return all_cvcl, breakdown_df


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
master_cvcl, cvcl_breakdown = build_master_cvcl_list(tables)

print("\n" + "=" * 50)
print(f"MASTER CVCL LIST: {len(master_cvcl):,} distinct CVCL- IDs total")
print("=" * 50)
print("\nPer-table breakdown:")
print(cvcl_breakdown.to_string(index=False))

master_cvcl_df = pd.DataFrame(
    sorted(master_cvcl), columns=["master_cvcl_id"]
)
print("\nSample of master CVCL list:")
print(master_cvcl_df.head())

hpa_rna_clean           :      0 distinct CVCL-
depmap_expr_clean       :      0 distinct CVCL-
geo_expr_clean          :      0 distinct CVCL-
proteomics_clean        :      0 distinct CVCL-
protein_map_clean       :      0 distinct CVCL-
fusions_clean           :      0 distinct CVCL-
mutations_clean         :      0 distinct CVCL-
cellosaurus_clean       : 152,231 distinct CVCL-
depmap_profiles_clean   :      0 distinct CVCL-
sample_info_clean       :  1,814 distinct CVCL-
geo_info_clean          :    797 distinct CVCL-
hpa_desc_clean          :  1,198 distinct CVCL-
metabolomics_clean      :      0 distinct CVCL-
mirna_clean             :      0 distinct CVCL-
signatures_clean        :      0 distinct CVCL-

MASTER CVCL LIST: 152,233 distinct CVCL- IDs total

Per-table breakdown:
                table  n_cvcl_found
    cellosaurus_clean        152231
    sample_info_clean          1814
       hpa_desc_clean          1198
       geo_info_clean           797
        hpa_rna_clean    

In [120]:
"""
=====================================================================
ADD master_cvcl_id COLUMN TO ALL TABLES
- Tables with CVCL- in an ID column -> fill from that column
- Tables with no CVCL-              -> empty (NA) column
=====================================================================
"""

import pandas as pd
import re

# ---------------------------------------------------------------
# Map: table -> source column holding the CVCL- id (None = empty col)
# ---------------------------------------------------------------
CVCL_SOURCE = {
    "cellosaurus_clean": "cellosaurus_accession",   # the CVCL- registry (primary)
    "sample_info_clean": "rrid",          # sample_info's CVCL- xref col
    "hpa_desc_clean":    "cellosaurus id",          # HPA's CVCL- link col
    # --- tables with NO direct CVCL- -> empty ---
    "depmap_profiles_clean": None,
    "fusions_clean":         None,
    "signatures_clean":      None,
    "metabolomics_clean":    None,
    "proteomics_clean":      None,
    "depmap_expr_clean":     None,
    "mutations_clean":       None,
    "geo_expr_clean":        None,
    "geo_info_clean":        "cellosaurus_id",
    "hpa_rna_clean":         None,
    "mirna_clean":           None,
}


# ---------------------------------------------------------------
# STEP 1: extraction helper
# ---------------------------------------------------------------
def extract_cvcl(series):
    """Extract bare lowercase CVCL- id from a column's values; NA where none."""
    return (
        series.astype(str)
        .str.extract(r"(cvcl_[a-z0-9]+)", flags=re.IGNORECASE)[0]
        .str.lower()
    )


# ---------------------------------------------------------------
# STEP 2: add master_cvcl_id to every table
# ---------------------------------------------------------------
def add_master_cvcl_id(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = CVCL_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_cvcl_id"] = extract_cvcl(df[source_col])
        else:
            df["master_cvcl_id"] = pd.NA
            # warn if we EXPECTED a column but it isn't there
            if source_col is not None:
                cvcl_like = [c for c in df.columns
                             if "cvcl" in str(c).lower()
                             or "cellosaurus" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"cvcl-like cols present: {cvcl_like}")

        updated[name] = df
        n_filled = df["master_cvcl_id"].notna().sum()
        print(f"{name:24s}: master_cvcl_id filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# ---------------------------------------------------------------
# RUN  (chain onto the existing tables dict — keeps gene + ach columns)
# ---------------------------------------------------------------
tables = add_master_cvcl_id(tables)

# assign by key (robust to extra entries in the dict)
hpa_rna_clean         = tables["hpa_rna_clean"]
depmap_expr_clean     = tables["depmap_expr_clean"]
geo_expr_clean        = tables["geo_expr_clean"]
proteomics_clean      = tables["proteomics_clean"]
fusions_clean         = tables["fusions_clean"]
mutations_clean       = tables["mutations_clean"]
cellosaurus_clean     = tables["cellosaurus_clean"]
depmap_profiles_clean = tables["depmap_profiles_clean"]
sample_info_clean     = tables["sample_info_clean"]
geo_info_clean        = tables["geo_info_clean"]
hpa_desc_clean        = tables["hpa_desc_clean"]
metabolomics_clean    = tables["metabolomics_clean"]
mirna_clean           = tables["mirna_clean"]
signatures_clean      = tables["signatures_clean"]


# ---------------------------------------------------------------
# STEP 3: verify against master_cvcl_df
# ---------------------------------------------------------------
print("\n" + "=" * 55)
print("VERIFICATION")
print("=" * 55)

print("All tables have master_cvcl_id column:",
      all("master_cvcl_id" in df.columns for df in tables.values()))

master_cvcl_set = set(master_cvcl_df["master_cvcl_id"].astype(str).str.lower()) \
    if "master_cvcl_id" in master_cvcl_df.columns \
    else set(master_cvcl_df.iloc[:, 0].astype(str).str.lower())

print("\nCVCL- IDs found in tables but NOT in master_cvcl_df:")
any_missing = False
for name, df in tables.items():
    found = set(df["master_cvcl_id"].dropna().astype(str))
    missing = found - master_cvcl_set
    if missing:
        any_missing = True
        print(f"  {name:24s}: {len(missing):>5,} not in master "
              f"(e.g. {list(missing)[:3]})")
if not any_missing:
    print("  none — every table's CVCL- IDs are covered by master_cvcl_df ✓")

hpa_rna_clean           : master_cvcl_id filled in         0 of 24,315,372 rows
depmap_expr_clean       : master_cvcl_id filled in         0 of    53,961 rows
geo_expr_clean          : master_cvcl_id filled in         0 of    19,914 rows
proteomics_clean        : master_cvcl_id filled in         0 of       375 rows
protein_map_clean       : master_cvcl_id filled in         0 of    12,558 rows
fusions_clean           : master_cvcl_id filled in         0 of   184,237 rows
mutations_clean         : master_cvcl_id filled in         0 of 1,066,869 rows
cellosaurus_clean       : master_cvcl_id filled in   152,231 of   152,231 rows
depmap_profiles_clean   : master_cvcl_id filled in         0 of     3,830 rows
sample_info_clean       : master_cvcl_id filled in     1,818 of     1,840 rows
geo_info_clean          : master_cvcl_id filled in     3,159 of     3,267 rows
hpa_desc_clean          : master_cvcl_id filled in     1,198 of     1,206 rows
metabolomics_clean      : master_cvcl_id filled in 

In [121]:
print("=" * 90)
print("FINAL STATE — all three spine columns on every table")
print("=" * 90)
print(f"{'table':<24} {'gene':<6} {'ach':<5} {'cvcl':<6} "
      f"{'gene_filled':>12} {'ach_filled':>12} {'cvcl_filled':>12}")
print("-" * 90)

for name, df in tables.items():
    has_gene = "master_ensembl_gene_id" in df.columns
    has_ach  = "master_ach_id" in df.columns
    has_cvcl = "master_cvcl_id" in df.columns

    gene_filled = df["master_ensembl_gene_id"].notna().sum() if has_gene else 0
    ach_filled  = df["master_ach_id"].notna().sum() if has_ach else 0
    cvcl_filled = df["master_cvcl_id"].notna().sum() if has_cvcl else 0

    print(f"{name:<24} {str(has_gene):<6} {str(has_ach):<5} {str(has_cvcl):<6} "
          f"{gene_filled:>12,} {ach_filled:>12,} {cvcl_filled:>12,}")

# overall flags
all_have_gene = all("master_ensembl_gene_id" in df.columns for df in tables.values())
all_have_ach  = all("master_ach_id" in df.columns for df in tables.values())
all_have_cvcl = all("master_cvcl_id" in df.columns for df in tables.values())

print("\nAll tables have master_ensembl_gene_id:", all_have_gene)
print("All tables have master_ach_id:          ", all_have_ach)
print("All tables have master_cvcl_id:         ", all_have_cvcl)

# combined: every table has all three
print("\nAll tables have ALL THREE spine columns:",
      all_have_gene and all_have_ach and all_have_cvcl)

FINAL STATE — all three spine columns on every table
table                    gene   ach   cvcl    gene_filled   ach_filled  cvcl_filled
------------------------------------------------------------------------------------------
hpa_rna_clean            True   True  True     24,315,372            0            0
depmap_expr_clean        True   True  True         53,961            0            0
geo_expr_clean           True   True  True         19,914            0            0
proteomics_clean         True   True  True              0          375            0
protein_map_clean        True   True  True              0            0            0
fusions_clean            True   True  True        184,084      184,237            0
mutations_clean          True   True  True      1,066,869            0            0
cellosaurus_clean        True   True  True              0            0      152,231
depmap_profiles_clean    True   True  True              0        3,830            0
sample_info_clea

In [122]:
"""
=====================================================================
MASTER PROFILE LIST — distinct PR- IDs from AUTHORITATIVE ID COLUMNS only
PR- IDs live in:
  - depmap_expr (index/sample_id), mutations (profileid),
    depmap_profiles (profileid), signatures (sequencingid)
=====================================================================
"""
import pandas as pd
import re

# Proper PR- key columns per table
PR_ID_COLUMNS = {
    "depmap_profiles_clean": ["profileid"],
    "mutations_clean":       ["profileid"],
    "signatures_clean":      ["sequencingid"],
    "depmap_expr_clean":     ["sample_id"],   # PR- in the sample_id column (transposed form)
}


def extract_pr_set(series):
    """Return a SET of distinct lowercase pr- ids from a column."""
    found = (
        series.dropna()
        .astype(str)
        .str.extract(r"(pr-[a-z0-9]+)", flags=re.IGNORECASE)[0]
        .dropna()
        .str.lower()
    )
    return set(found.unique())


def build_master_profile_list(tables: dict, verbose=True):
    all_pr = set()
    breakdown = []
    for name, df in tables.items():
        table_ids = set()
        for col in PR_ID_COLUMNS.get(name, []):
            if col in df.columns:
                table_ids |= extract_pr_set(df[col])
            elif verbose:
                cands = [c for c in df.columns
                         if "pr" in str(c).lower() or "profile" in str(c).lower()
                         or "sequencing" in str(c).lower() or "sample" in str(c).lower()]
                print(f"  ⚠ {name}: '{col}' not found. candidates: {cands}")
        all_pr |= table_ids
        breakdown.append({"table": name, "n_pr_found": len(table_ids)})
        if verbose:
            print(f"{name:24s}: {len(table_ids):>6,} distinct PR-")
    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_pr_found", ascending=False).reset_index(drop=True)
    return all_pr, breakdown_df


master_pr, pr_breakdown = build_master_profile_list(tables)

print("\n" + "=" * 50)
print(f"MASTER PROFILE LIST: {len(master_pr):,} distinct PR- IDs total")
print("=" * 50)
print("\nPer-table breakdown:")
print(pr_breakdown.to_string(index=False))

master_profile_df = pd.DataFrame(sorted(master_pr), columns=["master_pr_id"])
print("\nSample of master profile list:")
print(master_profile_df.head())

hpa_rna_clean           :      0 distinct PR-
  ⚠ depmap_expr_clean: 'sample_id' not found. candidates: ['pr-adbjpg', 'pr-i2azwg', 'pr-5ekaac', 'pr-i21681', 'pr-i9drp1', 'pr-llpkng', 'pr-fesgd6', 'pr-z36vet', 'pr-7wade1', 'pr-dpuwim', 'pr-o7ujdr', 'pr-i3qlza', 'pr-yrj8kp', 'pr-0hhxsp', 'pr-r9epko', 'pr-rmpave', 'pr-0hdbb5', 'pr-peuspj', 'pr-d95sz5', 'pr-koodhd', 'pr-f1yfes', 'pr-1jxoso', 'pr-arwg44', 'pr-l7jpvc', 'pr-x88vkn', 'pr-fhn7ap', 'pr-bkcagb', 'pr-daw5o4', 'pr-gckfgt', 'pr-uhihj8', 'pr-xpqgiv', 'pr-gbb5av', 'pr-2v3g3g', 'pr-0dhyny', 'pr-itqfdc', 'pr-atfy3k', 'pr-fvzpiv', 'pr-mbeeo0', 'pr-kibk5e', 'pr-uq6qid', 'pr-4rthc9', 'pr-gbulrr', 'pr-8cmbny', 'pr-xezhmi', 'pr-je4ebz', 'pr-gxtlhw', 'pr-yjfnfh', 'pr-px3fif', 'pr-vas5qc', 'pr-p5puml', 'pr-ovfjch', 'pr-slkhab', 'pr-orvtsc', 'pr-5jruuz', 'pr-0chffi', 'pr-84jv3m', 'pr-twe0mn', 'pr-2tdypf', 'pr-3dmyvz', 'pr-09gmei', 'pr-gywfk3', 'pr-mqzdb2', 'pr-gfsbbp', 'pr-hbv2dm', 'pr-dmyrk0', 'pr-jgwbfp', 'pr-zflvle', 'pr-hzhjza', 'pr-chspwa'

In [123]:
depmap_expr_clean

,gene,ensg_id,pr-adbjpg,pr-i2azwg,pr-5ekaac,pr-i21681,pr-i9drp1,pr-llpkng,pr-fesgd6,pr-z36vet,...,pr-iueft6,pr-heyoh9,pr-acnzor,pr-ez3iv8,pr-rwnv81,pr-ivfp8s,pr-asipq0,master_ensembl_gene_id,master_ach_id,master_cvcl_id
0,tspan6,ensg00000000003,4.331992,4.567424,3.150560,5.085340,6.729417,4.272770,3.337711,0.056584,...,5.995032,3.533563,0.056584,3.111031,4.390943,5.057450,4.249445,ensg00000000003,<NA>,<NA>
1,tnmd,ensg00000000005,0.000000,0.584963,0.000000,0.000000,0.000000,0.189034,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,ensg00000000005,<NA>,<NA>
2,dpm1,ensg00000000419,7.364660,7.106641,7.379118,7.154211,6.537917,7.023255,5.927659,6.094236,...,6.238978,6.488483,6.604368,7.031329,7.013239,7.815191,6.175724,ensg00000000419,<NA>,<NA>
3,scyl3,ensg00000000457,2.792855,2.543496,2.333424,2.545968,2.456806,2.555816,1.944858,3.971773,...,2.304511,1.823749,3.266037,1.541019,1.887525,2.538538,2.319040,ensg00000000457,<NA>,<NA>
4,c1orf112,ensg00000000460,4.471187,3.504620,4.228049,3.084064,3.867896,3.841973,2.678072,3.731183,...,4.000000,3.308885,4.973152,3.664483,3.252476,3.893362,3.825786,ensg00000000460,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53956,NaN,ensg00000288721,0.992768,0.432959,0.367371,0.411426,0.678072,0.189034,0.422233,1.304511,...,0.925999,0.879706,1.244887,0.454176,0.695994,0.604071,0.985500,ensg00000288721,<NA>,<NA>
53957,NaN,ensg00000288722,2.797013,2.972693,1.695994,3.921246,4.418190,3.054848,0.201634,5.596637,...,2.060047,4.978196,4.553975,5.377818,4.456806,4.196135,4.076388,ensg00000288722,<NA>,<NA>
53958,NaN,ensg00000288723,0.000000,0.056584,0.084064,0.028569,0.000000,0.000000,0.000000,0.124328,...,0.000000,0.028569,0.056584,0.310340,0.367371,0.084064,0.000000,ensg00000288723,<NA>,<NA>
53959,NaN,ensg00000288724,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,ensg00000288724,<NA>,<NA>


In [124]:
"""
=====================================================================
ADD master_pr_id COLUMN TO ALL TABLES
=====================================================================
"""
import pandas as pd
import re

PR_SOURCE = {
    "depmap_profiles_clean": "profileid",      # PR- -> ACH- bridge (PR- side)
    "mutations_clean":       "profileid",      # PR- keyed
    "signatures_clean":      "sequencingid",   # PR- keyed (second link)
    "depmap_expr_clean":     "sample_id",      # PR- in sample_id (transposed form)
    # --- tables with NO PR- -> empty ---
    "sample_info_clean":     None,
    "fusions_clean":         None,
    "metabolomics_clean":    None,
    "proteomics_clean":      None,
    "cellosaurus_clean":     None,
    "geo_expr_clean":        None,
    "geo_info_clean":        None,
    "hpa_rna_clean":         None,
    "hpa_desc_clean":        None,
    "mirna_clean":           None,
}


def extract_pr(series):
    """Extract bare lowercase PR- id; NA where none."""
    return (
        series.astype(str)
        .str.extract(r"(pr-[a-z0-9]+)", flags=re.IGNORECASE)[0]
        .str.lower()
    )


def add_master_pr_id(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = PR_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_pr_id"] = extract_pr(df[source_col])
        else:
            df["master_pr_id"] = pd.NA
            if source_col is not None:
                pr_like = [c for c in df.columns
                           if "pr" in str(c).lower() or "profile" in str(c).lower()
                           or "sequencing" in str(c).lower() or "sample" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"pr-like cols present: {pr_like}")

        updated[name] = df
        n_filled = df["master_pr_id"].notna().sum()
        print(f"{name:24s}: master_pr_id filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# RUN — chain onto the existing tables dict (keeps gene/ach/cvcl columns)
tables = add_master_pr_id(tables)

# reassign all variables at once
for name in tables:
    globals()[name] = tables[name]

hpa_rna_clean           : master_pr_id filled in         0 of 24,315,372 rows
  ⚠ depmap_expr_clean: expected column 'sample_id' not found. pr-like cols present: ['pr-adbjpg', 'pr-i2azwg', 'pr-5ekaac', 'pr-i21681', 'pr-i9drp1', 'pr-llpkng', 'pr-fesgd6', 'pr-z36vet', 'pr-7wade1', 'pr-dpuwim', 'pr-o7ujdr', 'pr-i3qlza', 'pr-yrj8kp', 'pr-0hhxsp', 'pr-r9epko', 'pr-rmpave', 'pr-0hdbb5', 'pr-peuspj', 'pr-d95sz5', 'pr-koodhd', 'pr-f1yfes', 'pr-1jxoso', 'pr-arwg44', 'pr-l7jpvc', 'pr-x88vkn', 'pr-fhn7ap', 'pr-bkcagb', 'pr-daw5o4', 'pr-gckfgt', 'pr-uhihj8', 'pr-xpqgiv', 'pr-gbb5av', 'pr-2v3g3g', 'pr-0dhyny', 'pr-itqfdc', 'pr-atfy3k', 'pr-fvzpiv', 'pr-mbeeo0', 'pr-kibk5e', 'pr-uq6qid', 'pr-4rthc9', 'pr-gbulrr', 'pr-8cmbny', 'pr-xezhmi', 'pr-je4ebz', 'pr-gxtlhw', 'pr-yjfnfh', 'pr-px3fif', 'pr-vas5qc', 'pr-p5puml', 'pr-ovfjch', 'pr-slkhab', 'pr-orvtsc', 'pr-5jruuz', 'pr-0chffi', 'pr-84jv3m', 'pr-twe0mn', 'pr-2tdypf', 'pr-3dmyvz', 'pr-09gmei', 'pr-gywfk3', 'pr-mqzdb2', 'pr-gfsbbp', 'pr-hbv2dm', 'pr-d

In [125]:
depmap_expr_clean

,gene,ensg_id,pr-adbjpg,pr-i2azwg,pr-5ekaac,pr-i21681,pr-i9drp1,pr-llpkng,pr-fesgd6,pr-z36vet,...,pr-heyoh9,pr-acnzor,pr-ez3iv8,pr-rwnv81,pr-ivfp8s,pr-asipq0,master_ensembl_gene_id,master_ach_id,master_cvcl_id,master_pr_id
0,tspan6,ensg00000000003,4.331992,4.567424,3.150560,5.085340,6.729417,4.272770,3.337711,0.056584,...,3.533563,0.056584,3.111031,4.390943,5.057450,4.249445,ensg00000000003,<NA>,<NA>,<NA>
1,tnmd,ensg00000000005,0.000000,0.584963,0.000000,0.000000,0.000000,0.189034,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,ensg00000000005,<NA>,<NA>,<NA>
2,dpm1,ensg00000000419,7.364660,7.106641,7.379118,7.154211,6.537917,7.023255,5.927659,6.094236,...,6.488483,6.604368,7.031329,7.013239,7.815191,6.175724,ensg00000000419,<NA>,<NA>,<NA>
3,scyl3,ensg00000000457,2.792855,2.543496,2.333424,2.545968,2.456806,2.555816,1.944858,3.971773,...,1.823749,3.266037,1.541019,1.887525,2.538538,2.319040,ensg00000000457,<NA>,<NA>,<NA>
4,c1orf112,ensg00000000460,4.471187,3.504620,4.228049,3.084064,3.867896,3.841973,2.678072,3.731183,...,3.308885,4.973152,3.664483,3.252476,3.893362,3.825786,ensg00000000460,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53956,NaN,ensg00000288721,0.992768,0.432959,0.367371,0.411426,0.678072,0.189034,0.422233,1.304511,...,0.879706,1.244887,0.454176,0.695994,0.604071,0.985500,ensg00000288721,<NA>,<NA>,<NA>
53957,NaN,ensg00000288722,2.797013,2.972693,1.695994,3.921246,4.418190,3.054848,0.201634,5.596637,...,4.978196,4.553975,5.377818,4.456806,4.196135,4.076388,ensg00000288722,<NA>,<NA>,<NA>
53958,NaN,ensg00000288723,0.000000,0.056584,0.084064,0.028569,0.000000,0.000000,0.000000,0.124328,...,0.028569,0.056584,0.310340,0.367371,0.084064,0.000000,ensg00000288723,<NA>,<NA>,<NA>
53959,NaN,ensg00000288724,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,ensg00000288724,<NA>,<NA>,<NA>


In [126]:
print("=" * 100)
print("FINAL STATE — all four spine columns on every table")
print("=" * 100)
print(f"{'table':<24} {'gene':>10} {'ach':>10} {'cvcl':>10} {'pr':>10}")
print("-" * 100)

for name, df in tables.items():
    g = df["master_ensembl_gene_id"].notna().sum() if "master_ensembl_gene_id" in df.columns else 0
    a = df["master_ach_id"].notna().sum()           if "master_ach_id" in df.columns else 0
    c = df["master_cvcl_id"].notna().sum()          if "master_cvcl_id" in df.columns else 0
    p = df["master_pr_id"].notna().sum()            if "master_pr_id" in df.columns else 0
    print(f"{name:<24} {g:>10,} {a:>10,} {c:>10,} {p:>10,}")

keys = ["master_ensembl_gene_id", "master_ach_id", "master_cvcl_id", "master_pr_id"]
print("\nAll tables have all four spine columns:",
      all(all(k in df.columns for k in keys) for df in tables.values()))

FINAL STATE — all four spine columns on every table
table                          gene        ach       cvcl         pr
----------------------------------------------------------------------------------------------------
hpa_rna_clean            24,315,372          0          0          0
depmap_expr_clean            53,961          0          0          0
geo_expr_clean               19,914          0          0          0
proteomics_clean                  0        375          0          0
protein_map_clean                 0          0          0          0
fusions_clean               184,084    184,237          0          0
mutations_clean           1,066,869          0          0  1,066,869
cellosaurus_clean                 0          0    152,231          0
depmap_profiles_clean             0      3,830          0      3,830
sample_info_clean                 0      1,840      1,818          0
geo_info_clean                    0          0      3,159          0
hpa_desc_clean     

In [127]:
"""
=====================================================================
MASTER GSM LIST — distinct GSM IDs from AUTHORITATIVE ID COLUMNS only
GSM IDs live in:
  - geo_expr  (GSM as column HEADERS in wide form, or sample_id if transposed)
  - geo_info  (geo_accession column)
=====================================================================
"""
import pandas as pd
import re


def extract_gsm_set_from_values(series):
    """Return a SET of distinct lowercase gsm ids from a column's values."""
    found = (
        series.dropna()
        .astype(str)
        .str.extract(r"(gsm\d+)", flags=re.IGNORECASE)[0]
        .dropna()
        .str.lower()
    )
    return set(found.unique())


def extract_gsm_set_from_headers(columns):
    """Return a SET of distinct lowercase gsm ids from column headers."""
    ids = set()
    pat = re.compile(r"gsm\d+", re.IGNORECASE)
    for c in columns:
        m = pat.search(str(c))
        if m:
            ids.add(m.group(0).lower())
    return ids


# table -> ("header" or "value", column or None)
GSM_ID_SOURCE = {
    "geo_info_clean": ("value", "geo_accession"),   # GSM in a column
    "geo_expr_clean": ("header", None),             # GSM in column headers (wide form)
}


def build_master_gsm_list(tables: dict, verbose=True):
    all_gsm = set()
    breakdown = []
    for name, df in tables.items():
        table_ids = set()
        if name in GSM_ID_SOURCE:
            loc_type, col = GSM_ID_SOURCE[name]
            if loc_type == "header":
                table_ids |= extract_gsm_set_from_headers(df.columns)
            elif col in df.columns:
                table_ids |= extract_gsm_set_from_values(df[col])
            elif verbose:
                cands = [c for c in df.columns
                         if "gsm" in str(c).lower() or "geo" in str(c).lower()
                         or "accession" in str(c).lower() or "sample" in str(c).lower()]
                print(f"  ⚠ {name}: '{col}' not found. candidates: {cands}")
        all_gsm |= table_ids
        breakdown.append({"table": name, "n_gsm_found": len(table_ids)})
        if verbose:
            print(f"{name:24s}: {len(table_ids):>6,} distinct GSM")
    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_gsm_found", ascending=False).reset_index(drop=True)
    return all_gsm, breakdown_df


master_gsm, gsm_breakdown = build_master_gsm_list(tables)

print("\n" + "=" * 50)
print(f"MASTER GSM LIST: {len(master_gsm):,} distinct GSM IDs total")
print("=" * 50)
print("\nPer-table breakdown:")
print(gsm_breakdown.to_string(index=False))

master_gsm_df = pd.DataFrame(sorted(master_gsm), columns=["master_gsm_id"])
print("\nSample of master GSM list:")
print(master_gsm_df.head())

hpa_rna_clean           :      0 distinct GSM
depmap_expr_clean       :      0 distinct GSM
geo_expr_clean          :  3,267 distinct GSM
proteomics_clean        :      0 distinct GSM
protein_map_clean       :      0 distinct GSM
fusions_clean           :      0 distinct GSM
mutations_clean         :      0 distinct GSM
cellosaurus_clean       :      0 distinct GSM
depmap_profiles_clean   :      0 distinct GSM
sample_info_clean       :      0 distinct GSM
geo_info_clean          :  3,267 distinct GSM
hpa_desc_clean          :      0 distinct GSM
metabolomics_clean      :      0 distinct GSM
mirna_clean             :      0 distinct GSM
signatures_clean        :      0 distinct GSM

MASTER GSM LIST: 3,267 distinct GSM IDs total

Per-table breakdown:
                table  n_gsm_found
       geo_expr_clean         3267
       geo_info_clean         3267
        hpa_rna_clean            0
    depmap_expr_clean            0
     proteomics_clean            0
    protein_map_clean          

In [128]:
"""
=====================================================================
ADD master_gsm_id COLUMN TO ALL TABLES
=====================================================================
"""
import pandas as pd
import re

# table -> source column holding GSM (None = empty col)
# NOTE: geo_expr in WIDE form has GSM in HEADERS, not a column -> can't fill a
#       per-row value, so it gets an empty column here (handled by header-scan
#       in the master list above instead). If geo_expr is TRANSPOSED (GSM in a
#       sample_id column), set its source to that column name.
GSM_SOURCE = {
    "geo_info_clean": "geo_accession",   # GSM keyed
    # "geo_expr_clean": "sample_id",     # uncomment IF geo_expr is transposed
    # --- tables with NO GSM -> empty ---
    "geo_expr_clean":        None,       # wide form: GSM in headers, not rows
    "sample_info_clean":     None,
    "depmap_profiles_clean": None,
    "fusions_clean":         None,
    "signatures_clean":      None,
    "metabolomics_clean":    None,
    "proteomics_clean":      None,
    "cellosaurus_clean":     None,
    "depmap_expr_clean":     None,
    "mutations_clean":       None,
    "hpa_rna_clean":         None,
    "hpa_desc_clean":        None,
    "mirna_clean":           None,
}


def extract_gsm(series):
    """Extract bare lowercase GSM id; NA where none."""
    return (
        series.astype(str)
        .str.extract(r"(gsm\d+)", flags=re.IGNORECASE)[0]
        .str.lower()
    )


def add_master_gsm_id(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = GSM_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_gsm_id"] = extract_gsm(df[source_col])
        else:
            df["master_gsm_id"] = pd.NA
            if source_col is not None:
                gsm_like = [c for c in df.columns
                            if "gsm" in str(c).lower() or "geo" in str(c).lower()
                            or "accession" in str(c).lower() or "sample" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"gsm-like cols present: {gsm_like}")

        updated[name] = df
        n_filled = df["master_gsm_id"].notna().sum()
        print(f"{name:24s}: master_gsm_id filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# RUN — chain onto existing tables dict (keeps gene/ach/cvcl/pr columns)
tables = add_master_gsm_id(tables)

# reassign all variables at once
for name in tables:
    globals()[name] = tables[name]

hpa_rna_clean           : master_gsm_id filled in         0 of 24,315,372 rows
depmap_expr_clean       : master_gsm_id filled in         0 of    53,961 rows
geo_expr_clean          : master_gsm_id filled in         0 of    19,914 rows
proteomics_clean        : master_gsm_id filled in         0 of       375 rows
protein_map_clean       : master_gsm_id filled in         0 of    12,558 rows
fusions_clean           : master_gsm_id filled in         0 of   184,237 rows
mutations_clean         : master_gsm_id filled in         0 of 1,066,869 rows
cellosaurus_clean       : master_gsm_id filled in         0 of   152,231 rows
depmap_profiles_clean   : master_gsm_id filled in         0 of     3,830 rows
sample_info_clean       : master_gsm_id filled in         0 of     1,840 rows
geo_info_clean          : master_gsm_id filled in     3,267 of     3,267 rows
hpa_desc_clean          : master_gsm_id filled in         0 of     1,206 rows
metabolomics_clean      : master_gsm_id filled in         0 of 

In [129]:
print("=" * 110)
print("FINAL STATE — all five spine columns on every table")
print("=" * 110)
print(f"{'table':<24} {'gene':>10} {'ach':>10} {'cvcl':>10} {'pr':>10} {'gsm':>10}")
print("-" * 110)

for name, df in tables.items():
    g = df["master_ensembl_gene_id"].notna().sum() if "master_ensembl_gene_id" in df.columns else 0
    a = df["master_ach_id"].notna().sum()           if "master_ach_id" in df.columns else 0
    c = df["master_cvcl_id"].notna().sum()          if "master_cvcl_id" in df.columns else 0
    p = df["master_pr_id"].notna().sum()            if "master_pr_id" in df.columns else 0
    s = df["master_gsm_id"].notna().sum()           if "master_gsm_id" in df.columns else 0
    print(f"{name:<24} {g:>10,} {a:>10,} {c:>10,} {p:>10,} {s:>10,}")

keys = ["master_ensembl_gene_id", "master_ach_id", "master_cvcl_id",
        "master_pr_id", "master_gsm_id"]
print("\nAll tables have all five spine columns:",
      all(all(k in df.columns for k in keys) for df in tables.values()))

FINAL STATE — all five spine columns on every table
table                          gene        ach       cvcl         pr        gsm
--------------------------------------------------------------------------------------------------------------
hpa_rna_clean            24,315,372          0          0          0          0
depmap_expr_clean            53,961          0          0          0          0
geo_expr_clean               19,914          0          0          0          0
proteomics_clean                  0        375          0          0          0
protein_map_clean                 0          0          0          0          0
fusions_clean               184,084    184,237          0          0          0
mutations_clean           1,066,869          0          0  1,066,869          0
cellosaurus_clean                 0          0    152,231          0          0
depmap_profiles_clean             0      3,830          0      3,830          0
sample_info_clean                 0  

In [130]:
"""
=====================================================================
MASTER CELL LINE NAME LIST — distinct names from AUTHORITATIVE name columns
Name columns vary per table; values need normalisation (names are messy:
case, hyphens, tissue suffixes). This harvests the raw distinct names.
=====================================================================
"""
import pandas as pd
import re

# Proper cell-line-NAME columns per table (not IDs)
NAME_ID_COLUMNS = {
    "sample_info_clean": ["stripped_cell_line_name"],   # bare DepMap name
    "hpa_rna_clean":     ["cell line"],                 # HPA name
    "hpa_desc_clean":    ["cell line"],                 # HPA desc name
    "cellosaurus_clean": ["cellosaurus_cell_line_name"],# Cellosaurus name
    "metabolomics_clean":["ccle_id"], 
    "geo_info_clean":["cellline"],                  # CCLE-style name (w/ tissue)
    # mirna names are in COLUMN HEADERS (ccle-style) -> handled separately
}


def clean_name(series):
    """Normalise names: lowercase, strip, collapse spaces."""
    return (
        series.dropna().astype(str)
        .str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
    )


def build_master_cellline_name_list(tables: dict, verbose=True):
    all_names = set()
    breakdown = []
    for name, df in tables.items():
        table_names = set()
        for col in NAME_ID_COLUMNS.get(name, []):
            if col in df.columns:
                table_names |= set(clean_name(df[col]).unique())
            elif verbose:
                cands = [c for c in df.columns
                         if "name" in str(c).lower() or "cell" in str(c).lower()
                         or "ccle" in str(c).lower()]
                print(f"  ⚠ {name}: '{col}' not found. candidates: {cands}")

        # mirna: names are column headers (drop the 2 non-name cols)
        if name == "mirna_clean":
            header_names = {str(c).strip().lower() for c in df.columns} - {"name", "description"}
            table_names |= header_names

        all_names |= table_names
        breakdown.append({"table": name, "n_names_found": len(table_names)})
        if verbose:
            print(f"{name:24s}: {len(table_names):>6,} distinct names")

    breakdown_df = pd.DataFrame(breakdown).sort_values(
        "n_names_found", ascending=False).reset_index(drop=True)
    return all_names, breakdown_df


master_names, name_breakdown = build_master_cellline_name_list(tables)

print("\n" + "=" * 50)
print(f"MASTER CELL LINE NAME LIST: {len(master_names):,} distinct names total")
print("=" * 50)
print("\nPer-table breakdown:")
print(name_breakdown.to_string(index=False))

master_name_df = pd.DataFrame(sorted(master_names), columns=["master_cellline_name"])
print("\nSample:")
print(master_name_df.head())

hpa_rna_clean           :  1,205 distinct names
depmap_expr_clean       :      0 distinct names
geo_expr_clean          :      0 distinct names
proteomics_clean        :      0 distinct names
protein_map_clean       :      0 distinct names
fusions_clean           :      0 distinct names
mutations_clean         :      0 distinct names
cellosaurus_clean       : 151,675 distinct names
depmap_profiles_clean   :      0 distinct names
  ⚠ sample_info_clean: 'stripped_cell_line_name' not found. candidates: ['cell_line_name', 'ccle_name', 'wtsi_master_cell_id', 'cellosaurus_ncit_disease', 'cellosaurus_ncit_id', 'cellosaurus_issues']
sample_info_clean       :      0 distinct names
geo_info_clean          :    802 distinct names
hpa_desc_clean          :  1,205 distinct names
metabolomics_clean      :    928 distinct names
mirna_clean             :    959 distinct names
signatures_clean        :      0 distinct names

MASTER CELL LINE NAME LIST: 152,701 distinct names total

Per-table breakdown:

In [131]:
"""
=====================================================================
ADD master_cellline_name COLUMN TO ALL TABLES
=====================================================================
"""
import pandas as pd
import re

NAME_SOURCE = {
    "sample_info_clean":  "cell_line_name",
    "hpa_rna_clean":      "cell line",
    "hpa_desc_clean":     "cell line",
    "cellosaurus_clean":  "cellosaurus_cell_line_name",
    "metabolomics_clean": "cell_line_name",
    # --- no row-level name -> empty ---
    "depmap_profiles_clean": None,
    "fusions_clean":         None,
    "signatures_clean":      None,
    "proteomics_clean":      None,
    "depmap_expr_clean":     None,
    "mutations_clean":       None,
    "geo_expr_clean":        None,
    "geo_info_clean":        "cellline",   # the clean GEO name col (if present)
    "mirna_clean":           None,                  # names in headers, not rows
}


def clean_name_series(series):
    """Normalise a name column; NA stays NA."""
    s = series.astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
    return s.replace({"nan": pd.NA, "<na>": pd.NA, "": pd.NA})


def add_master_cellline_name(tables: dict):
    updated = {}
    for name, df in tables.items():
        df = df.copy()
        source_col = NAME_SOURCE.get(name, None)

        if source_col is not None and source_col in df.columns:
            df["master_cellline_name"] = clean_name_series(df[source_col])
        else:
            df["master_cellline_name"] = pd.NA
            if source_col is not None:
                cands = [c for c in df.columns
                         if "name" in str(c).lower() or "cell" in str(c).lower()
                         or "ccle" in str(c).lower()]
                print(f"  ⚠ {name}: expected column '{source_col}' not found. "
                      f"name-like cols present: {cands}")

        updated[name] = df
        n_filled = df["master_cellline_name"].notna().sum()
        print(f"{name:24s}: master_cellline_name filled in {n_filled:>9,} of {len(df):>9,} rows")

    return updated


# RUN — chain onto existing tables dict
tables = add_master_cellline_name(tables)
for name in tables:
    globals()[name] = tables[name]

hpa_rna_clean           : master_cellline_name filled in 24,315,372 of 24,315,372 rows
depmap_expr_clean       : master_cellline_name filled in         0 of    53,961 rows
geo_expr_clean          : master_cellline_name filled in         0 of    19,914 rows
proteomics_clean        : master_cellline_name filled in         0 of       375 rows
protein_map_clean       : master_cellline_name filled in         0 of    12,558 rows
fusions_clean           : master_cellline_name filled in         0 of   184,237 rows
mutations_clean         : master_cellline_name filled in         0 of 1,066,869 rows
cellosaurus_clean       : master_cellline_name filled in   152,230 of   152,231 rows
depmap_profiles_clean   : master_cellline_name filled in         0 of     3,830 rows
sample_info_clean       : master_cellline_name filled in     1,748 of     1,840 rows
geo_info_clean          : master_cellline_name filled in     2,681 of     3,267 rows
hpa_desc_clean          : master_cellline_name filled in     1,

In [132]:
sample_info_clean

,depmap_id,cell_line_name,ccle_name,alias,cosmicid,sex,source,rrid,wtsi_master_cell_id,sample_collection_site,...,parent_depmap_id,cellosaurus_ncit_disease,cellosaurus_ncit_id,cellosaurus_issues,master_ensembl_gene_id,master_ach_id,master_cvcl_id,master_pr_id,master_gsm_id,master_cellline_name
0,ach-000016,slr21,slr21_kidney,NaN,NaN,NaN,academic lab,cvcl_v607,NaN,kidney,...,NaN,clear cell renal cell carcinoma,c4033,NaN,<NA>,ach-000016,cvcl_v607,<NA>,<NA>,slr21
1,ach-000032,mhhcall3,mhhcall3_haematopoietic_and_lymphoid_tissue,NaN,NaN,female,dsmz,cvcl_0089,NaN,bone_marrow,...,NaN,childhood b acute lymphoblastic leukemia,c9140,NaN,<NA>,ach-000032,cvcl_0089,<NA>,<NA>,mhhcall3
2,ach-000033,ncih1819,ncih1819_lung,NaN,NaN,female,academic lab,cvcl_1497,NaN,lymph_node,...,NaN,lung adenocarcinoma,c3512,NaN,<NA>,ach-000033,cvcl_1497,<NA>,<NA>,ncih1819
3,ach-000043,hs895t,hs895t_fibroblast,NaN,NaN,female,atcc,cvcl_0993,NaN,fibroblast,...,NaN,melanoma,c3224,NaN,<NA>,ach-000043,cvcl_0993,<NA>,<NA>,hs895t
4,ach-000049,hekte,hekte_kidney,NaN,NaN,NaN,academic lab,cvcl_ws59,NaN,kidney,...,NaN,NaN,NaN,no information is available about this cell li...,<NA>,ach-000049,cvcl_ws59,<NA>,<NA>,hekte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1835,ach-002393,croap3,croap3_haematopoietic_and_lymphoid_tissue,NaN,NaN,male,sanger,cvcl_1810,NaN,ascites,...,NaN,primary effusion lymphoma,c6915,NaN,<NA>,ach-002393,cvcl_1810,<NA>,<NA>,croap3
1836,ach-002394,geo,geo_large_intestine,NaN,NaN,NaN,sanger,cvcl_0271,NaN,large_intestine,...,NaN,colon carcinoma,c4910,NaN,<NA>,ach-002394,cvcl_0271,<NA>,<NA>,geo
1837,ach-002395,huh6clone5,huh6clone5_liver,NaN,NaN,male,sanger,cvcl_1296,NaN,liver,...,ach-000671,hepatoblastoma,c3728,NaN,<NA>,ach-002395,cvcl_1296,<NA>,<NA>,huh6clone5
1838,ach-002396,sarc9371,sarc9371_bone,NaN,NaN,NaN,sanger,cvcl_5g89,NaN,bone,...,NaN,osteosarcoma,c9145,NaN,<NA>,ach-002396,cvcl_5g89,<NA>,<NA>,sarc9371


In [133]:
print("=" * 125)
print("FINAL STATE — all six spine columns on every table")
print("=" * 125)
hdr = f"{'table':<24} {'gene':>9} {'ach':>9} {'cvcl':>9} {'pr':>9} {'gsm':>9} {'name':>9}"
print(hdr); print("-" * 125)

cols = {
    "gene": "master_ensembl_gene_id", "ach": "master_ach_id",
    "cvcl": "master_cvcl_id", "pr": "master_pr_id",
    "gsm": "master_gsm_id", "celline_name": "master_cellline_name",
}
for name, df in tables.items():
    vals = [df[c].notna().sum() if c in df.columns else 0 for c in cols.values()]
    print(f"{name:<24} " + " ".join(f"{v:>9,}" for v in vals))

print("\nAll tables have all six spine columns:",
      all(all(c in df.columns for c in cols.values()) for df in tables.values()))

FINAL STATE — all six spine columns on every table
table                         gene       ach      cvcl        pr       gsm      name
-----------------------------------------------------------------------------------------------------------------------------
hpa_rna_clean            24,315,372         0         0         0         0 24,315,372
depmap_expr_clean           53,961         0         0         0         0         0
geo_expr_clean              19,914         0         0         0         0         0
proteomics_clean                 0       375         0         0         0         0
protein_map_clean                0         0         0         0         0         0
fusions_clean              184,084   184,237         0         0         0         0
mutations_clean          1,066,869         0         0 1,066,869         0         0
cellosaurus_clean                0         0   152,231         0         0   152,230
depmap_profiles_clean            0     3,830         0  

In [134]:
# sample_info has the ID + several name columns side by side
name_cols = [c for c in sample_info_clean.columns
             if "name" in c.lower() or "ccle" in c.lower() or "alias" in c.lower()]

print("ID column: depmap_id (ach-)")
print("Name columns found:", name_cols)
print()

# show one cell line's ID alongside its various names
sample = sample_info_clean[["depmap_id"] + name_cols].head(10)
print(sample.to_string(index=False))

ID column: depmap_id (ach-)
Name columns found: ['cell_line_name', 'ccle_name', 'alias', 'master_cellline_name']

 depmap_id cell_line_name                                   ccle_name alias master_cellline_name
ach-000016          slr21                                slr21_kidney   NaN                slr21
ach-000032       mhhcall3 mhhcall3_haematopoietic_and_lymphoid_tissue   NaN             mhhcall3
ach-000033       ncih1819                               ncih1819_lung   NaN             ncih1819
ach-000043         hs895t                           hs895t_fibroblast   NaN               hs895t
ach-000049          hekte                                hekte_kidney   NaN                hekte
ach-000051         te617t                          te617t_soft_tissue   NaN               te617t
ach-000064           sale                                   sale_lung   NaN                 sale
ach-000068           rec1     rec1_haematopoietic_and_lymphoid_tissue   NaN                 rec1
ach-000071   

In [135]:
# more robust: strip only known tissue suffixes from the END
TISSUE_SUFFIXES = (
    sample_info_clean["ccle_name"].astype(str)
    .str.extract(r"_([a-z_]+)$")[0].dropna().value_counts()
)
print("Tissue suffixes found in ccle_name (most common):")
print(TISSUE_SUFFIXES.head(20))

Tissue suffixes found in ccle_name (most common):
0
haematopoietic_and_lymphoid_tissue    285
lung                                  244
skin                                  111
central_nervous_system                110
breast                                 87
large_intestine                        81
upper_aerodigestive_tract              78
bone                                   76
soft_tissue                            75
ovary                                  74
pancreas                               59
kidney                                 58
autonomic_ganglia                      49
stomach                                48
biliary_tract                          43
fibroblast                             39
matched_normal_tissue                  39
oesophagus                             38
urinary_tract                          38
endometrium                            37
Name: count, dtype: int64


In [136]:
hpa_rna_clean

,gene,gene name,cell line,tpm,ptpm,ntpm,master_ensembl_gene_id,master_ach_id,master_cvcl_id,master_pr_id,master_gsm_id,master_cellline_name
0,ensg00000000003,tspan6,143b,22.0,27.6,25.9,ensg00000000003,<NA>,<NA>,<NA>,<NA>,143b
1,ensg00000000003,tspan6,22rv1,2.8,3.6,2.7,ensg00000000003,<NA>,<NA>,<NA>,<NA>,22rv1
2,ensg00000000003,tspan6,23132/87,6.2,7.5,7.5,ensg00000000003,<NA>,<NA>,<NA>,<NA>,23132/87
3,ensg00000000003,tspan6,253j,14.2,18.7,25.4,ensg00000000003,<NA>,<NA>,<NA>,<NA>,253j
4,ensg00000000003,tspan6,253jbv,13.0,17.1,18.5,ensg00000000003,<NA>,<NA>,<NA>,<NA>,253jbv
...,...,...,...,...,...,...,...,...,...,...,...,...
24315367,ensg00000291317,tmem276,yh13,16.8,21.8,15.6,ensg00000291317,<NA>,<NA>,<NA>,<NA>,yh13
24315368,ensg00000291317,tmem276,ykg1,25.1,32.2,24.0,ensg00000291317,<NA>,<NA>,<NA>,<NA>,ykg1
24315369,ensg00000291317,tmem276,ymb1,42.4,55.9,47.8,ensg00000291317,<NA>,<NA>,<NA>,<NA>,ymb1
24315370,ensg00000291317,tmem276,zr751,30.0,40.9,30.8,ensg00000291317,<NA>,<NA>,<NA>,<NA>,zr751
